<a href="https://colab.research.google.com/github/SreeGayatri369/AI-Meeting-Assitant/blob/main/Welcome_To_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install -U gradio requests
import sys, gradio
print("Python", sys.version.split()[0], "| Gradio", gradio.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
Python 3.13.15 | Gradio 6.29.0


In [5]:
import os, re, json, glob, time, shutil, smtplib, tempfile, subprocess, datetime, html, requests
from email.message import EmailMessage

esc = html.escape
KEYS = {}  # filled in Cell 3. Server-side only, never sent to the browser.

def sec(name):
    try:
        from google.colab import userdata
        return userdata.get(name) or ""
    except Exception:
        return os.environ.get(name, "")

class ApiError(Exception):
    pass

# ---------- HTTP helper: retries + readable errors ----------
def _msg(r):
    try:
        j = r.json()
        if isinstance(j, list) and j: j = j[0]
        e = j.get("error", j) if isinstance(j, dict) else j
        return (e.get("message") if isinstance(e, dict) else str(e)) or r.text[:300]
    except Exception:
        return r.text[:300]

HINTS = {401: "Invalid or missing API key.", 403: "Access denied for this key.",
         404: "Model or endpoint not found - check the model name.",
         413: "Input too large for this model/plan.", 429: "Rate limit reached."}

def _retry_after(r, default):
    try: return float(r.headers.get("retry-after", default))
    except Exception: return default

def post(url, label, retries=3, **kw):
    for n in range(retries + 1):
        try:
            r = requests.post(url, timeout=300, **kw)
        except requests.exceptions.RequestException as e:
            if n < retries:
                time.sleep(3 * (n + 1)); continue
            raise ApiError(f"{label}: network problem ({type(e).__name__}). Check your connection and retry.")
        if r.status_code in (429, 500, 502, 503) and n < retries:
            wait = _retry_after(r, 10 * (n + 1))
            if r.status_code == 429 and wait > 90:
                raise ApiError(f"{label}: rate limit reached - try again in ~{int(wait // 60) + 1} min. {_msg(r)}")
            time.sleep(min(wait, 90)); continue
        if not r.ok:
            raise ApiError(f"{label} error {r.status_code}: {HINTS.get(r.status_code, '')} {_msg(r)}".strip())
        return r

def friendly(e):
    if isinstance(e, ApiError): return str(e)
    if isinstance(e, requests.exceptions.RequestException):
        return f"Network problem ({type(e).__name__}). Check your connection and retry."
    if isinstance(e, smtplib.SMTPAuthenticationError):
        return "Gmail rejected the login. Use a 16-character App Password (needs 2-step verification), not your normal password."
    if isinstance(e, smtplib.SMTPException): return f"Email error: {e}"
    return f"Unexpected error: {type(e).__name__}: {e}"

# ---------- Transcription (Groq Whisper API) + hallucination filtering ----------
GROQ_STT = "https://api.groq.com/openai/v1/audio/transcriptions"
CHUNK_SEC = 600
HALLU = {"thank you", "thanks", "thank you very much", "thanks for watching", "thank you for watching",
         "please subscribe", "bye", "goodbye", "bye bye", "you", "okay", "ok"}

def norm(t): return re.sub(r"[^\w\s]", "", t.lower()).strip()
def is_hallu(n):
    return n in HALLU or n.startswith(("thanks for watching", "thank you for watching")) or "amaraorg" in n

def silences(path, db=-45, dur=1.0):
    """Silent spans (start, end) in seconds. Lower db (e.g. -50) = less aggressive."""
    p = subprocess.run(["ffmpeg", "-hide_banner", "-nostats", "-i", path, "-vn", "-af",
                        f"silencedetect=noise={db}dB:d={dur}", "-f", "null", "-"], capture_output=True, text=True)
    out, start = [], None
    for line in p.stderr.splitlines():
        m = re.search(r"silence_start:\s*(-?[\d.]+)", line)
        if m: start = max(0.0, float(m.group(1))); continue
        m = re.search(r"silence_end:\s*([\d.]+)", line)
        if m and start is not None: out.append((start, float(m.group(1)))); start = None
    if start is not None: out.append((start, 1e9))
    return out

def silent_fraction(a, b, spans):
    d = max(b - a, 1e-3)
    return min(1.0, sum(max(0, min(b, y) - max(a, x)) for x, y in spans) / d)

def clean_segments(segs, spans):
    kept = []
    for s in segs:
        text = s["text"].strip()
        if not text: continue
        if s["nsp"] > 0.6 and s["lp"] < -1.0: continue   # Whisper's own "no speech" rule
        if s["cr"] > 2.4: continue                       # repetitive gibberish
        s["n"] = norm(text); s["sil"] = silent_fraction(s["start"], s["end"], spans)
        if s["sil"] >= 0.9: continue
        if s["sil"] >= 0.5 and (is_hallu(s["n"]) or s["nsp"] > 0.3): continue
        if is_hallu(s["n"]) and s["nsp"] > 0.5: continue
        kept.append(s)
    out, run = [], 0                                     # repeat filter: max 2 identical lines in a row
    for s in kept:
        run = run + 1 if out and s["n"] == out[-1]["n"] else 1
        if run <= 2: out.append(s)
    j = len(out)                                         # trailing "thank you" style hallucinations
    while j > 0 and is_hallu(out[j - 1]["n"]): j -= 1
    tail = out[j:]
    if len(tail) >= 2 or (len(tail) == 1 and (tail[0]["sil"] > 0.3 or tail[0]["nsp"] > 0.3)):
        out = out[:j]
    return out

def format_segments(segs):
    return "\n".join(f"[{int(s['start'])//3600:02d}:{int(s['start'])%3600//60:02d}:{int(s['start'])%60:02d}] {s['text'].strip()}"
                     for s in segs)

def transcribe_groq(path, gkey, model, language, cb=lambda f, m: None):
    if not gkey: raise ApiError("No Groq API key. Add it in Settings (free: console.groq.com/keys).")
    if not shutil.which("ffmpeg"): raise ApiError("ffmpeg missing. Run  !apt-get -qq install ffmpeg  and retry.")
    tmp = tempfile.mkdtemp()
    try:
        p = subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", path, "-vn", "-ac", "1", "-ar", "16000",
                            "-b:a", "32k", "-f", "segment", "-segment_time", str(CHUNK_SEC),
                            "-reset_timestamps", "1", f"{tmp}/part_%03d.mp3"], capture_output=True, text=True)
        parts = sorted(glob.glob(f"{tmp}/part_*.mp3"))
        if p.returncode != 0 or not parts:
            raise ApiError("Could not read audio from this file (no audio track or unsupported format). " + p.stderr[-200:])
        cb(0.05, "Detecting silence...")
        spans = silences(path)
        segs = []
        for i, part in enumerate(parts):
            cb(0.1 + 0.6 * i / len(parts), f"Transcribing part {i + 1}/{len(parts)}...")
            data = {"model": model, "response_format": "verbose_json", "temperature": 0}
            if language != "auto": data["language"] = language
            with open(part, "rb") as fh: blob = fh.read()
            r = post(GROQ_STT, "Groq transcription", headers={"Authorization": f"Bearer {gkey}"},
                     files={"file": (os.path.basename(part), blob, "audio/mpeg")}, data=data)
            for s in r.json().get("segments", []):
                segs.append({"start": i * CHUNK_SEC + s.get("start", 0), "end": i * CHUNK_SEC + s.get("end", 0),
                             "text": s.get("text", ""), "nsp": s.get("no_speech_prob") or 0,
                             "lp": s.get("avg_logprob") or 0, "cr": s.get("compression_ratio") or 0})
        cb(0.72, "Cleaning transcript...")
        return format_segments(clean_segments(segs, spans))
    finally:
        shutil.rmtree(tmp, ignore_errors=True)

# ---------- LLM (direct HTTP, no openai package) ----------
PROVIDERS = {
    "Groq":   {"url": "https://api.groq.com/openai/v1", "model": "llama-3.3-70b-versatile"},
    "OpenAI": {"url": "https://api.openai.com/v1", "model": "gpt-4o-mini"},
    "Gemini": {"url": "https://generativelanguage.googleapis.com/v1beta/openai", "model": "gemini-2.5-flash"},
}

def chat(provider, key, model, system, user, json_mode=False):
    if not key: raise ApiError(f"No API key for {provider}.")
    url = PROVIDERS[provider]["url"].rstrip("/") + "/chat/completions"
    hdr = {"Authorization": f"Bearer {key}", "Content-Type": "application/json"}
    body = {"model": model, "temperature": 0.2,
            "messages": [{"role": "system", "content": system}, {"role": "user", "content": user}]}
    if json_mode: body["response_format"] = {"type": "json_object"}
    try:
        r = post(url, provider, headers=hdr, json=body)
    except ApiError as e:
        if json_mode and " 400" in str(e):               # model without JSON mode: retry plain
            body.pop("response_format"); r = post(url, provider, headers=hdr, json=body)
        else:
            raise
    try:
        return r.json()["choices"][0]["message"]["content"] or ""
    except Exception:
        raise ApiError(f"{provider}: unexpected response format.")

SYSTEM = """You are an expert meeting assistant. Return ONLY valid JSON with keys:
"title" (short string), "summary" (3-6 sentence string), "key_points" (list of strings),
"decisions" (list of strings), "action_items" (list of {"task","owner","due"}; use "Unassigned"/"N/A" if unknown),
"open_questions" (list of strings), "followup_email" (polite ready-to-send email body string).
Never invent facts that are not in the transcript."""

def parse_json(raw):
    raw = re.sub(r"^```(?:json)?|```$", "", raw.strip(), flags=re.M).strip()
    try: return json.loads(raw)
    except Exception:
        m = re.search(r"\{.*\}", raw, re.S)
        if m:
            try: return json.loads(m.group(0))
            except Exception: pass
    raise ApiError("The model did not return valid JSON. Click Regenerate or pick another model.")

def as_list(x): return [] if x is None else list(x) if isinstance(x, (list, tuple)) else [x]
def as_text(x): return " - ".join(str(v) for v in x.values() if v) if isinstance(x, dict) else str(x or "").strip()

def clean_result(r):
    acts = []
    for a in as_list(r.get("action_items")):
        if isinstance(a, dict):
            acts.append({"task": as_text(a.get("task") or a.get("action")), "owner": as_text(a.get("owner")) or "Unassigned",
                         "due": as_text(a.get("due")) or "N/A"})
        elif str(a).strip():
            acts.append({"task": str(a).strip(), "owner": "Unassigned", "due": "N/A"})
    L = lambda k: [as_text(x) for x in as_list(r.get(k)) if x]
    return {"title": as_text(r.get("title")) or "Meeting Notes", "summary": as_text(r.get("summary")),
            "key_points": L("key_points"), "decisions": L("decisions"), "open_questions": L("open_questions"),
            "action_items": [a for a in acts if a["task"]], "followup_email": as_text(r.get("followup_email"))}

def condense(text, provider, key, model):
    for _ in range(3):
        if len(text) <= 22000: return text
        text = "\n\n".join(chat(provider, key, model,
            "Condense this part of a meeting transcript to under 2500 characters. Keep decisions, action items (who/what/when), names, numbers.",
            text[i:i + 16000]) for i in range(0, len(text), 16000))
    return text[:22000]

def analyze_text(transcript, provider, key, model):
    text = condense(transcript, provider, key, model)
    return clean_result(parse_json(chat(provider, key, model, SYSTEM, text, json_mode=True)))

def pick_context(transcript, question, limit=22000):
    if len(transcript) <= limit: return transcript
    lines = transcript.splitlines()
    blocks = ["\n".join(lines[i:i + 12]) for i in range(0, len(lines), 12)]
    words = set(re.findall(r"\w{3,}", question.lower()))
    order = sorted(range(len(blocks)), key=lambda i: -len(words & set(re.findall(r"\w{3,}", blocks[i].lower()))))
    chosen, total = [], 0
    for i in order:
        if total + len(blocks[i]) <= limit: chosen.append(i); total += len(blocks[i])
    return "\n...\n".join(blocks[i] for i in sorted(chosen)) or blocks[0][:limit]

# ---------- Rendering: cards + markdown ----------
CARD = ("background:var(--background-fill-secondary);border:1px solid var(--border-color-primary);"
        "border-radius:14px;padding:16px 18px;margin:10px 0;color:var(--body-text-color)")
EMPTY = '<div style="opacity:.6;padding:12px">Notes will appear here.</div>'

def chip(t, bg): return f'<span style="background:{bg};border-radius:999px;padding:2px 10px;font-size:12px;margin-right:6px;white-space:nowrap">{esc(t)}</span>'
def card(icon, head, body): return f'<div style="{CARD}"><div style="font-weight:600;font-size:15px;margin-bottom:8px">{icon} {esc(head)}</div>{body}</div>'
def bullets(xs):
    if not xs: return '<div style="opacity:.6">None</div>'
    return '<ul style="margin:0;padding-left:20px;line-height:1.65">' + "".join(f"<li>{esc(x)}</li>" for x in xs) + "</ul>"

def render_cards(r):
    acts = "".join(
        '<div style="display:flex;gap:10px;align-items:flex-start;padding:8px 0;border-top:1px solid var(--border-color-primary)">'
        '<div style="width:16px;height:16px;border:2px solid #6366f1;border-radius:5px;margin-top:3px;flex:none"></div>'
        f'<div><div style="font-weight:500">{esc(a["task"])}</div><div style="margin-top:4px">'
        f'{chip("👤 " + a["owner"], "rgba(99,102,241,.18)")}{chip("📅 " + a["due"], "rgba(245,158,11,.22)")}</div></div></div>'
        for a in r["action_items"]) or '<div style="opacity:.6">None</div>'
    return (f'<div><div style="font-size:22px;font-weight:700">{esc(r["title"])}</div>'
            f'<div style="opacity:.65;font-size:13px">{datetime.date.today():%d %b %Y} · {len(r["action_items"])} action items · {len(r["decisions"])} decisions</div></div>'
            + card("📝", "Summary", f'<div style="line-height:1.65">{esc(r["summary"])}</div>')
            + card("✅", "Action items", acts) + card("🏁", "Decisions", bullets(r["decisions"]))
            + card("💡", "Key points", bullets(r["key_points"])) + card("❓", "Open questions", bullets(r["open_questions"]))
            + f'<details style="{CARD}"><summary style="cursor:pointer;font-weight:600">✉️ Follow-up email</summary>'
              f'<pre style="white-space:pre-wrap;font-family:inherit;margin:10px 0 0">{esc(r["followup_email"])}</pre></details>')

def to_markdown(r):
    L = lambda xs: "\n".join(f"- {x}" for x in xs) or "- None"
    acts = "\n".join(f"- [ ] {a['task']} - **{a['owner']}** (due: {a['due']})" for a in r["action_items"]) or "- None"
    return (f"# {r['title']}\n_{datetime.date.today():%d %b %Y}_\n\n## Summary\n{r['summary']}\n\n## Action Items\n{acts}\n\n"
            f"## Decisions\n{L(r['decisions'])}\n\n## Key Points\n{L(r['key_points'])}\n\n"
            f"## Open Questions\n{L(r['open_questions'])}\n\n## Follow-up Email\n{r['followup_email']}\n")

# ---------- Integrations ----------
def notion_call(method, url, token, **kw):
    r = requests.request(method, url, headers={"Authorization": f"Bearer {token}", "Notion-Version": "2022-06-28"}, timeout=60, **kw)
    if not r.ok:
        hint = {401: "Invalid Notion token.", 404: "Not found - share the page/database with your integration (⋯ → Connections)."}.get(r.status_code, "")
        raise ApiError(f"Notion error {r.status_code}: {hint} {_msg(r)}".strip())
    return r.json()

def push_notion(token, target, kind, title, md):
    if not token or not target: raise ApiError("Notion: add the integration token and page/database ID in Settings.")
    ids = re.findall(r"[0-9a-fA-F]{32}", (target.split("?")[0]).replace("-", ""))
    if not ids: raise ApiError("Notion: could not find a 32-character ID in what you pasted.")
    tid = ids[-1]
    rt = lambda t: [{"type": "text", "text": {"content": t[:1900]}}]
    blocks = []
    for line in md.splitlines():
        s = line.strip().replace("**", "")
        if len(s) > 2 and s[0] == "_" and s[-1] == "_": s = s[1:-1]
        if not s or s.startswith("# "): continue
        if s.startswith("## "): blocks.append({"type": "heading_2", "heading_2": {"rich_text": rt(s[3:])}})
        elif s.startswith("- [ ] "): blocks.append({"type": "to_do", "to_do": {"rich_text": rt(s[6:]), "checked": False}})
        elif s.startswith("- "): blocks.append({"type": "bulleted_list_item", "bulleted_list_item": {"rich_text": rt(s[2:])}})
        else:
            for i in range(0, len(s), 1900):
                blocks.append({"type": "paragraph", "paragraph": {"rich_text": rt(s[i:i + 1900])}})
    if kind == "Database":
        db = notion_call("GET", f"https://api.notion.com/v1/databases/{tid}", token)
        tp = next((k for k, v in db.get("properties", {}).items() if v.get("type") == "title"), None)
        if not tp: raise ApiError("Notion: that ID is not a database. Choose 'Page' as target type.")
        parent, props = {"database_id": tid}, {tp: {"title": rt(title)}}
    else:
        parent, props = {"page_id": tid}, {"title": {"title": rt(title)}}
    page = notion_call("POST", "https://api.notion.com/v1/pages", token,
                       json={"parent": parent, "properties": props, "children": blocks[:100]})
    for i in range(100, len(blocks), 100):
        notion_call("PATCH", f"https://api.notion.com/v1/blocks/{page['id']}/children", token, json={"children": blocks[i:i + 100]})
    return page.get("url", "created")

def push_slack(webhook, md):
    if not webhook: raise ApiError("Slack: add the webhook URL in Settings.")
    text = re.sub(r"^#+\s*(.*)$", r"*\1*", md, flags=re.M).replace("**", "*")[:35000]
    r = requests.post(webhook, json={"text": text}, timeout=30)
    if r.text.strip() != "ok": raise ApiError(f"Slack error {r.status_code}: {r.text[:200]}")
    return "sent"

def send_email(user, pw, to, subject, body):
    if not (user and pw and to): raise ApiError("Email: fill Gmail address, app password and recipient.")
    m = EmailMessage(); m["From"], m["To"], m["Subject"] = user, to, subject; m.set_content(body)
    with smtplib.SMTP_SSL("smtp.gmail.com", 465, timeout=30) as s:
        s.login(user, pw.replace(" ", "")); s.send_message(m)
    return f"sent to {to}"

In [10]:
# ---- Cell 2b (replacement): self-healing model selection ----
PROVIDERS["Groq"]["model"] = "openai/gpt-oss-120b"
PREFS = {
    "Groq":   ["openai/gpt-oss-120b", "openai/gpt-oss-20b", "qwen/qwen3.8-27b", "qwen/qwen3.6-27b", "llama-3.3-70b-versatile"],
    "OpenAI": ["gpt-4o-mini", "gpt-4.1-mini", "gpt-4o"],
    "Gemini": ["gemini-2.5-flash", "gemini-2.0-flash", "gemini-flash-latest"],
}
_SKIP = ("whisper", "guard", "tts", "embed", "orpheus", "image", "moderation", "transcribe", "audio", "safeguard", "dall", "realtime", "compound")
_MODELS, _WORKING = {}, {}

def list_models(provider, key, refresh=False):
    ck = (provider, key[-8:])
    if refresh or not _MODELS.get(ck):
        ids = []
        try:
            r = requests.get(PROVIDERS[provider]["url"].rstrip("/") + "/models",
                             headers={"Authorization": f"Bearer {key}"}, timeout=30)
            if r.ok:
                ids = [m["id"].replace("models/", "") for m in r.json().get("data", [])]
        except Exception:
            pass
        _MODELS[ck] = ids
    return _MODELS[ck]

def candidates(provider, key, requested):
    ids = list_models(provider, key)
    chat_ids = [i for i in ids if not any(s in i.lower() for s in _SKIP)]
    order = [_WORKING.get(provider), requested] + PREFS.get(provider, []) + chat_ids
    out = []
    for m in order:
        if m and m not in out and (not ids or m in ids):
            out.append(m)
    return (out or [requested])[:6]

def _model_problem(e):
    s = str(e).lower()
    return bool(re.search(r" (400|403|404)", s)) and any(w in s for w in ("model", "not found", "decommission", "access", "permission"))

def chat(provider, key, model, system, user, json_mode=False):
    if not key: raise ApiError(f"No API key for {provider}.")
    url = PROVIDERS[provider]["url"].rstrip("/") + "/chat/completions"
    hdr = {"Authorization": f"Bearer {key}", "Content-Type": "application/json"}
    last = None
    for m in candidates(provider, key, model):
        body = {"model": m, "temperature": 0,
                "messages": [{"role": "system", "content": system}, {"role": "user", "content": user}]}
        if json_mode: body["response_format"] = {"type": "json_object"}
        try:
            try:
                r = post(url, provider, headers=hdr, json=body)
            except ApiError as e:
                if json_mode and " 400" in str(e) and any(w in str(e).lower() for w in ("json", "response_format")):
                    body.pop("response_format"); r = post(url, provider, headers=hdr, json=body)
                else:
                    raise
        except ApiError as e:
            last = e
            if _model_problem(e):
                print(f"[info] {provider} model '{m}' unavailable -> trying next")
                continue
            raise
        if _WORKING.get(provider) != m:
            _WORKING[provider] = m; print(f"[info] using {provider} model: {m}")
        try:
            text = r.json()["choices"][0]["message"]["content"] or ""
        except Exception:
            raise ApiError(f"{provider}: unexpected response format.")
        return re.sub(r"<think>.*?</think>", "", text, flags=re.S).strip()
    ids = list_models(provider, key)
    raise ApiError(f"{provider}: none of the models worked ({last}). Models your key can see: "
                   f"{', '.join(ids[:15]) or 'none - the key may be invalid or the endpoint unreachable'}")

def condense(text, provider, key, model):          # smaller chunks for free-tier limits
    for _ in range(4):
        if len(text) <= 14000: return text
        text = "\n\n".join(chat(provider, key, model,
            "Condense this part of a meeting transcript to under 2000 characters. Keep decisions, action items (who/what/when), names, numbers.",
            text[i:i + 10000]) for i in range(0, len(text), 10000))
    return text[:14000]

_old_pick = pick_context
def pick_context(transcript, question, limit=12000):
    return _old_pick(transcript, question, limit)

print("Model patch applied. Default Groq model:", PROVIDERS["Groq"]["model"])

Model patch applied. Default Groq model: openai/gpt-oss-120b


In [11]:
from getpass import getpass

for k in ["GROQ_KEY", "LLM_KEY", "NOTION_TOKEN", "NOTION_ID", "SLACK_WEBHOOK", "SMTP_USER", "SMTP_PASS", "APP_PASSWORD"]:
    KEYS[k] = sec(k)
if not KEYS["GROQ_KEY"]:
    KEYS["GROQ_KEY"] = getpass("Groq API key (hidden, free at console.groq.com/keys) - Enter to skip: ").strip()

print("ffmpeg :", "OK" if shutil.which("ffmpeg") else "MISSING -> run: !apt-get -qq install ffmpeg")
if KEYS["GROQ_KEY"]:
    try:
        out = chat("Groq", KEYS["GROQ_KEY"], PROVIDERS["Groq"]["model"], "Reply with the single word OK.", "ping")
        print("Groq chat completions : OK ->", out.strip()[:20])
    except Exception as e:
        print("Groq chat completions : FAILED ->", friendly(e))
else:
    print("No Groq key entered - you can paste it inside the UI instead.")

Groq API key (hidden, free at console.groq.com/keys) - Enter to skip: ··········
ffmpeg : OK
Groq chat completions : FAILED -> Unexpected error: RecursionError: maximum recursion depth exceeded


In [9]:
# ---- Patch 2c: strictly transcript-grounded notes ----
_post_prev = post
if not getattr(_post_prev, "_t0", False):
    def post(url, label, retries=3, **kw):          # force temperature 0 for all LLM calls
        if isinstance(kw.get("json"), dict) and "temperature" in kw["json"]:
            kw["json"]["temperature"] = 0
        return _post_prev(url, label, retries, **kw)
    post._t0 = True

TS_RE = re.compile(r"^\[(\d{1,2}:\d{2}:\d{2})\]\s*(.*)$")
WORD_RE = re.compile(r"[A-Za-z0-9][\w+#./'’-]*")
_UNITS = {w: i for i, w in enumerate("zero one two three four five six seven eight nine ten eleven twelve thirteen fourteen fifteen sixteen seventeen eighteen nineteen".split())}
_TENS = {"twenty": 20, "thirty": 30, "forty": 40, "fifty": 50, "sixty": 60, "seventy": 70, "eighty": 80, "ninety": 90}
OPENERS = set("""the a an it its they their he she we our this that these those there then also next first second third finally lastly
overall however additionally later earlier initially meanwhile afterward subsequently in on at after before during as when while according
several some one two three many most all both each speaker speakers discussion topics participants attendees group team meeting conversation
another other no not here if because since although though for from with without by to of about regarding concerning throughout among between
instead otherwise still yet so but and or now today i ok okay""".split())
STOP = set("that this with from have will would should could about there their were been they them then than what when which while also into over such just very more some".split())
_NONE = {"", "unassigned", "n/a", "na", "none", "unknown", "not specified", "not stated", "tbd"}

def _n(t): return " ".join(re.findall(r"[a-z0-9']+", t.lower().replace("’", "'")))
def _tok(t):
    t = t.lower().replace("’", "'").strip(".,;:!?)('\"")
    return t[:-2] if t.endswith("'s") else t

class Transcript:
    def __init__(self, text):
        self.lines = []
        for raw in text.splitlines():
            s = raw.strip()
            if not s: continue
            m = TS_RE.match(s)
            ts, t = (m.group(1), m.group(2)) if m else (None, s)
            self.lines.append((ts, t, _n(t)))
        self.has_ts = any(l[0] for l in self.lines)
        self.vocab = set()
        for _, t, _x in self.lines:
            self.vocab.update(_tok(w) for w in WORD_RE.findall(t))
        words = " ".join(l[2] for l in self.lines).split()
        for i, w in enumerate(words):                  # spoken numbers -> digits ("twenty five" -> 25)
            if w in _UNITS: self.vocab.add(str(_UNITS[w]))
            if w in _TENS:
                self.vocab.add(str(_TENS[w]))
                if i + 1 < len(words) and 0 < _UNITS.get(words[i + 1], 0) < 10:
                    self.vocab.add(str(_TENS[w] + _UNITS[words[i + 1]]))

    def locate(self, quote, hint=None):
        """Index of the transcript line containing the quote (verbatim, or >=85% word overlap), else None."""
        q = _n(re.sub(r"\[\d{1,2}:\d{2}:\d{2}\]", " ", quote or ""))
        qt = q.split()
        if len(qt) < 3: return None
        L = self.lines
        hits = [i for i, l in enumerate(L) if q in l[2]]
        if not hits:
            hits = [i for i in range(len(L) - 1) if q in L[i][2] + " " + L[i + 1][2]]
        if not hits and len(set(qt)) >= 4:
            qs = set(qt)
            for i in range(len(L)):
                win = set((L[i][2] + " " + (L[i + 1][2] if i + 1 < len(L) else "")).split())
                if len(qs & win) / len(qs) >= 0.85: hits.append(i)
        if not hits: return None
        m = re.search(r"\d{1,2}:\d{2}:\d{2}", hint or "")
        if m:
            for i in hits:
                if (L[i][0] or "").zfill(8) == m.group(0).zfill(8): return i
        return hits[0]

    def ctx(self, i, n=2):
        return " ".join(l[1] for l in self.lines[max(0, i - n): i + n + 2])

    def ungrounded(self, s, strict_first=False):
        """True if the text has a name / acronym / number that never appears in the transcript."""
        for k, t in enumerate(WORD_RE.findall(s)):
            n = _tok(t)
            if not n or n in OPENERS: continue
            dig = any(c.isdigit() for c in n)
            chk = dig or (t[0].isupper() if k else (strict_first and t[0].isupper()))
            if chk and n not in self.vocab:
                if "-" in n and all(p in self.vocab for p in n.split("-") if p): continue
                return True
        return False

    def filter_text(self, text):
        out, removed = [], 0
        for para in re.split(r"\n\s*\n", text or ""):
            keep = []
            for s in re.split(r"(?<=[.!?])\s+", para.strip()):
                if not s.strip(): continue
                if self.ungrounded(s, True): removed += 1
                else: keep.append(s.strip())
            if keep: out.append(" ".join(keep))
        return "\n\n".join(out), removed

def _supported(text, ctx_text):
    ws = [w for w in re.findall(r"[a-z0-9']{4,}", text.lower()) if w not in STOP]
    if not ws: return True
    cv = set(_n(ctx_text).split())
    hit = sum(1 for w in ws if w in cv or any(len(c) >= 4 and (c.startswith(w[:5]) or w.startswith(c[:5])) for c in cv))
    return hit / len(ws) >= 0.3

def _action_extra(it, tr, i):
    cs = set(_n(tr.ctx(i, 3)).split())
    owner, due = as_text(it.get("owner")), as_text(it.get("due"))
    ow, dw = _n(owner).split(), _n(due).split()
    if owner.lower() in _NONE or not ow or not all(w in cs for w in ow): owner = "Unassigned"
    if due.lower() in _NONE or not dw or not all(w in cs for w in dw): due = "N/A"
    return {"owner": owner, "due": due}

HEDGE = re.compile(r"\b(maybe|perhaps|might|possibly|someday|should we|could we|what if|if we|wish|hopefully)\b", re.I)

def _take(tr, parts, key, field, extra=None, firm=False):
    out, seen, bad = [], set(), 0
    for p in parts:
        for it in as_list(p.get(key)):
            if not isinstance(it, dict): bad += 1; continue
            text, quote = as_text(it.get(field)), as_text(it.get("quote"))
            i = tr.locate(quote, quote) if text and quote else None
            if i is None or tr.ungrounded(text) or not _supported(text, tr.ctx(i, 2)):
                bad += 1; continue
            if firm and HEDGE.search(tr.lines[i][1]): bad += 1; continue   # hedged talk is not a decision/commitment
            k = i if key == "action_items" else _n(text)[:90]               # one action item per transcript line
            if k in seen: continue
            seen.add(k)
            d = {"text": text, "ts": (tr.lines[i][0] if tr.has_ts else None) or "no timestamp", "quote": tr.lines[i][1][:220]}
            if extra: d.update(extra(it, tr, i))
            out.append(d)
    return out, bad

def _chunks(tr, limit=10000):
    out, cur, n = [], [], 0
    for ts, t, _x in tr.lines:
        line = f"[{ts}] {t}" if ts else t
        if cur and n + len(line) > limit: out.append("\n".join(cur)); cur, n = [], 0
        cur.append(line); n += len(line) + 1
    if cur: out.append("\n".join(cur))
    return out

CHUNK_SYS = """You are a meticulous, literal meeting note-taker. Use ONLY information explicitly stated in the transcript excerpt.
NEVER infer, guess, assume, generalize, or use outside knowledge. Do not add any person names, roles, job titles, companies, products,
tools, technologies, numbers, dates or deadlines that are not written in the excerpt. If speakers are not named, do not invent names;
say "the speaker" or "the group". If something is unclear, leave it out.

Return ONLY valid JSON with these keys:
"title": short neutral title in sentence case, using only words/topics from the excerpt.
"summary": a DETAILED factual summary of 2-4 paragraphs separated by blank lines (about 6-14 sentences, fewer if the excerpt is short).
  Cover every topic in the order discussed, including the specifics that were actually stated (figures, names, reasons, problems, options).
  Every sentence must be directly supported by the excerpt. Begin sentences with neutral openers such as "The speaker", "The group",
  "It was mentioned that", "They", "The discussion" - never with a person's name.
"key_points": [{"point": str, "quote": str}]
"decisions": [{"decision": str, "quote": str}] - ONLY something explicitly agreed, decided or approved ("we decided", "let's go with", "agreed").
  Ideas, suggestions, options, opinions, or "we should/might" are NOT decisions.
"action_items": [{"task": str, "owner": str, "due": str, "quote": str}] - ONLY when someone is clearly assigned a task or commits to it
  ("I will...", "X will...", "Can you...?" followed by agreement). owner = a name only if stated in the excerpt, else "Unassigned".
  due = the deadline exactly as spoken, else "N/A". Wishes, ideas, problems or general discussion are NOT action items.
"open_questions": [{"question": str, "quote": str}] - questions explicitly raised and left unresolved.
Every "quote" must be copied VERBATIM (max 25 words) from a single transcript line and must start with that line's [HH:MM:SS] timestamp.
If you cannot give a verbatim quote, omit the item. Empty lists are correct and expected when nothing qualifies. Never pad."""

FINAL_SYS = """You are given factual notes from consecutive parts of ONE meeting. Merge them into one detailed, coherent summary of 3-6 paragraphs
separated by blank lines. Use ONLY facts present in the notes: add nothing, infer nothing, keep names and numbers exactly as written.
Begin sentences with neutral openers ("The speaker", "The group", "It was mentioned that", "They") - never with a person's name.
Return ONLY JSON: {"title": short sentence-case title using only words from the notes, "summary": str}."""

def _email(r):
    if not r["summary"] and not (r["decisions"] or r["action_items"]): return ""
    L = ["Hi all,", "", "Thank you for joining. Here is a recap of what was discussed.", "", r["summary"]]
    if r["decisions"]: L += ["", "Decisions:"] + [f"- {d['text']} [{d['ts']}]" for d in r["decisions"]]
    if r["action_items"]: L += ["", "Action items:"] + [f"- {a['text']} (owner: {a['owner']}, due: {a['due']}) [{a['ts']}]" for a in r["action_items"]]
    if r["open_questions"]: L += ["", "Open questions:"] + [f"- {q['text']} [{q['ts']}]" for q in r["open_questions"]]
    return "\n".join(L + ["", "Best regards"])

def analyze_text(transcript, provider, key, model):
    tr = Transcript(transcript)
    if not tr.lines: raise ApiError("Transcript is empty.")
    parts = [parse_json(chat(provider, key, model, CHUNK_SYS, "TRANSCRIPT EXCERPT:\n" + ch, json_mode=True)) for ch in _chunks(tr)]
    sums = [as_text(p.get("summary")) for p in parts]
    summary, title = "\n\n".join(s for s in sums if s), as_text(parts[0].get("title"))
    joined = "\n\n".join(f"Part {i + 1}:\n{s}" for i, s in enumerate(sums) if s)
    if len(parts) > 1 and 0 < len(joined) <= 14000:
        try:
            m = parse_json(chat(provider, key, model, FINAL_SYS, joined, json_mode=True))
            if as_text(m.get("summary")): summary, title = as_text(m["summary"]), as_text(m.get("title")) or title
        except Exception:
            pass                                       # fall back to the concatenated (already grounded) part summaries
    summary, removed = tr.filter_text(summary)
    if not title or tr.ungrounded(title): title = "Meeting notes"
    pts, b1 = _take(tr, parts, "key_points", "point")
    dec, b2 = _take(tr, parts, "decisions", "decision", firm=True)
    act, b3 = _take(tr, parts, "action_items", "task", _action_extra, firm=True)
    qs, b4 = _take(tr, parts, "open_questions", "question")
    r = {"title": title, "summary": summary or "No verifiable summary could be produced from this transcript.",
         "key_points": pts, "decisions": dec, "action_items": act, "open_questions": qs, "removed": removed + b1 + b2 + b3 + b4}
    r["followup_email"] = _email(r)
    return r

# ---------- Rendering (same look, now with evidence) ----------
def _ev(x): return f'<div style="opacity:.65;font-size:12px;margin-top:2px">⏱ {esc(x["ts"])} · “{esc(x["quote"])}”</div>'
def _items(xs, empty="Nothing explicit in the transcript."):
    if not xs: return f'<div style="opacity:.6">{empty}</div>'
    return "".join(f'<div style="padding:7px 0;border-top:1px solid var(--border-color-primary)"><div>{esc(x["text"])}</div>{_ev(x)}</div>' for x in xs)

def render_cards(r):
    acts = "".join(
        '<div style="display:flex;gap:10px;align-items:flex-start;padding:8px 0;border-top:1px solid var(--border-color-primary)">'
        '<div style="width:16px;height:16px;border:2px solid #6366f1;border-radius:5px;margin-top:3px;flex:none"></div>'
        f'<div><div style="font-weight:500">{esc(a["text"])}</div><div style="margin-top:4px">'
        f'{chip("👤 " + a["owner"], "rgba(99,102,241,.18)")}{chip("📅 " + a["due"], "rgba(245,158,11,.22)")}</div>{_ev(a)}</div></div>'
        for a in r["action_items"]) or '<div style="opacity:.6">No clear assignment or commitment found in the transcript.</div>'
    paras = "".join(f'<p style="margin:0 0 10px;line-height:1.65">{esc(p)}</p>' for p in r["summary"].split("\n\n"))
    note = (f'🔒 Grounding check: {r["removed"]} unverifiable item(s)/sentence(s) were removed.' if r["removed"]
            else "🔒 Grounding check passed: every item is backed by a transcript quote.")
    return (f'<div><div style="font-size:22px;font-weight:700">{esc(r["title"])}</div>'
            f'<div style="opacity:.65;font-size:13px">{datetime.date.today():%d %b %Y} · {len(r["action_items"])} action items · {len(r["decisions"])} decisions</div></div>'
            + card("📝", "Summary", paras) + card("✅", "Action items", acts)
            + card("🏁", "Decisions", _items(r["decisions"])) + card("💡", "Key points", _items(r["key_points"]))
            + card("❓", "Open questions", _items(r["open_questions"], "None raised."))
            + f'<details style="{CARD}"><summary style="cursor:pointer;font-weight:600">✉️ Follow-up email</summary>'
              f'<pre style="white-space:pre-wrap;font-family:inherit;margin:10px 0 0">{esc(r["followup_email"])}</pre></details>'
            + f'<div style="opacity:.6;font-size:12px;margin-top:8px">{note}</div>')

def to_markdown(r):
    dec = "\n".join(f"- {d['text']} [{d['ts']}] — “{d['quote']}”" for d in r["decisions"]) or "- None stated"
    act = "\n".join(f"- [ ] {a['text']} - **{a['owner']}** (due: {a['due']}) [{a['ts']}] — “{a['quote']}”" for a in r["action_items"]) or "- None stated"
    kp = "\n".join(f"- {k['text']} [{k['ts']}]" for k in r["key_points"]) or "- None"
    oq = "\n".join(f"- {q['text']} [{q['ts']}]" for q in r["open_questions"]) or "- None"
    return (f"# {r['title']}\n_{datetime.date.today():%d %b %Y}_\n\n## Summary\n{r['summary']}\n\n## Action Items\n{act}\n\n"
            f"## Decisions\n{dec}\n\n## Key Points\n{kp}\n\n## Open Questions\n{oq}\n\n## Follow-up Email\n{r['followup_email']}\n")

print("Grounded pipeline patch applied.")

Grounded pipeline patch applied.


In [ ]:
# ---- Patch 2c: strictly transcript-grounded notes ----
_post_prev = post
if not getattr(_post_prev, "_t0", False):
    def post(url, label, retries=3, **kw):          # force temperature 0 for all LLM calls
        if isinstance(kw.get("json"), dict) and "temperature" in kw["json"]:
            kw["json"]["temperature"] = 0
        return _post_prev(url, label, retries, **kw)
    post._t0 = True

TS_RE = re.compile(r"^\[(\d{1,2}:\d{2}:\d{2})\]\s*(.*)$")
WORD_RE = re.compile(r"[A-Za-z0-9][\w+#./'’-]*")
_UNITS = {w: i for i, w in enumerate("zero one two three four five six seven eight nine ten eleven twelve thirteen fourteen fifteen sixteen seventeen eighteen nineteen".split())}
_TENS = {"twenty": 20, "thirty": 30, "forty": 40, "fifty": 50, "sixty": 60, "seventy": 70, "eighty": 80, "ninety": 90}
OPENERS = set("""the a an it its they their he she we our this that these those there then also next first second third finally lastly
overall however additionally later earlier initially meanwhile afterward subsequently in on at after before during as when while according
several some one two three many most all both each speaker speakers discussion topics participants attendees group team meeting conversation
another other no not here if because since although though for from with without by to of about regarding concerning throughout among between
instead otherwise still yet so but and or now today i ok okay""".split())
STOP = set("that this with from have will would should could about there their were been they them then than what when which while also into over such just very more some".split())
_NONE = {"", "unassigned", "n/a", "na", "none", "unknown", "not specified", "not stated", "tbd"}

def _n(t): return " ".join(re.findall(r"[a-z0-9']+", t.lower().replace("’", "'")))
def _tok(t):
    t = t.lower().replace("’", "'").strip(".,;:!?)('\"")
    return t[:-2] if t.endswith("'s") else t

class Transcript:
    def __init__(self, text):
        self.lines = []
        for raw in text.splitlines():
            s = raw.strip()
            if not s: continue
            m = TS_RE.match(s)
            ts, t = (m.group(1), m.group(2)) if m else (None, s)
            self.lines.append((ts, t, _n(t)))
        self.has_ts = any(l[0] for l in self.lines)
        self.vocab = set()
        for _, t, _x in self.lines:
            self.vocab.update(_tok(w) for w in WORD_RE.findall(t))
        words = " ".join(l[2] for l in self.lines).split()
        for i, w in enumerate(words):                  # spoken numbers -> digits ("twenty five" -> 25)
            if w in _UNITS: self.vocab.add(str(_UNITS[w]))
            if w in _TENS:
                self.vocab.add(str(_TENS[w]))
                if i + 1 < len(words) and 0 < _UNITS.get(words[i + 1], 0) < 10:
                    self.vocab.add(str(_TENS[w] + _UNITS[words[i + 1]]))

    def locate(self, quote, hint=None):
        """Index of the transcript line containing the quote (verbatim, or >=85% word overlap), else None."""
        q = _n(re.sub(r"\[\d{1,2}:\d{2}:\d{2}\]", " ", quote or ""))
        qt = q.split()
        if len(qt) < 3: return None
        L = self.lines
        hits = [i for i, l in enumerate(L) if q in l[2]]
        if not hits:
            hits = [i for i in range(len(L) - 1) if q in L[i][2] + " " + L[i + 1][2]]
        if not hits and len(set(qt)) >= 4:
            qs = set(qt)
            for i in range(len(L)):
                win = set((L[i][2] + " " + (L[i + 1][2] if i + 1 < len(L) else "")).split())
                if len(qs & win) / len(qs) >= 0.85: hits.append(i)
        if not hits: return None
        m = re.search(r"\d{1,2}:\d{2}:\d{2}", hint or "")
        if m:
            for i in hits:
                if (L[i][0] or "").zfill(8) == m.group(0).zfill(8): return i
        return hits[0]

    def ctx(self, i, n=2):
        return " ".join(l[1] for l in self.lines[max(0, i - n): i + n + 2])

    def ungrounded(self, s, strict_first=False):
        """True if the text has a name / acronym / number that never appears in the transcript."""
        for k, t in enumerate(WORD_RE.findall(s)):
            n = _tok(t)
            if not n or n in OPENERS: continue
            dig = any(c.isdigit() for c in n)
            chk = dig or (t[0].isupper() if k else (strict_first and t[0].isupper()))
            if chk and n not in self.vocab:
                if "-" in n and all(p in self.vocab for p in n.split("-") if p): continue
                return True
        return False

    def filter_text(self, text):
        out, removed = [], 0
        for para in re.split(r"\n\s*\n", text or ""):
            keep = []
            for s in re.split(r"(?<=[.!?])\s+", para.strip()):
                if not s.strip(): continue
                if self.ungrounded(s, True): removed += 1
                else: keep.append(s.strip())
            if keep: out.append(" ".join(keep))
        return "\n\n".join(out), removed

def _supported(text, ctx_text):
    ws = [w for w in re.findall(r"[a-z0-9']{4,}", text.lower()) if w not in STOP]
    if not ws: return True
    cv = set(_n(ctx_text).split())
    hit = sum(1 for w in ws if w in cv or any(len(c) >= 4 and (c.startswith(w[:5]) or w.startswith(c[:5])) for c in cv))
    return hit / len(ws) >= 0.3

def _action_extra(it, tr, i):
    cs = set(_n(tr.ctx(i, 3)).split())
    owner, due = as_text(it.get("owner")), as_text(it.get("due"))
    ow, dw = _n(owner).split(), _n(due).split()
    if owner.lower() in _NONE or not ow or not all(w in cs for w in ow): owner = "Unassigned"
    if due.lower() in _NONE or not dw or not all(w in cs for w in dw): due = "N/A"
    return {"owner": owner, "due": due}

HEDGE = re.compile(r"\b(maybe|perhaps|might|possibly|someday|should we|could we|what if|if we|wish|hopefully)\b", re.I)

def _take(tr, parts, key, field, extra=None, firm=False):
    out, seen, bad = [], set(), 0
    for p in parts:
        for it in as_list(p.get(key)):
            if not isinstance(it, dict): bad += 1; continue
            text, quote = as_text(it.get(field)), as_text(it.get("quote"))
            i = tr.locate(quote, quote) if text and quote else None
            if i is None or tr.ungrounded(text) or not _supported(text, tr.ctx(i, 2)):
                bad += 1; continue
            if firm and HEDGE.search(tr.lines[i][1]): bad += 1; continue   # hedged talk is not a decision/commitment
            k = i if key == "action_items" else _n(text)[:90]               # one action item per transcript line
            if k in seen: continue
            seen.add(k)
            d = {"text": text, "ts": (tr.lines[i][0] if tr.has_ts else None) or "no timestamp", "quote": tr.lines[i][1][:220]}
            if extra: d.update(extra(it, tr, i))
            out.append(d)
    return out, bad

def _chunks(tr, limit=10000):
    out, cur, n = [], [], 0
    for ts, t, _x in tr.lines:
        line = f"[{ts}] {t}" if ts else t
        if cur and n + len(line) > limit: out.append("\n".join(cur)); cur, n = [], 0
        cur.append(line); n += len(line) + 1
    if cur: out.append("\n".join(cur))
    return out

CHUNK_SYS = """You are a meticulous, literal meeting note-taker. Use ONLY information explicitly stated in the transcript excerpt.
NEVER infer, guess, assume, generalize, or use outside knowledge. Do not add any person names, roles, job titles, companies, products,
tools, technologies, numbers, dates or deadlines that are not written in the excerpt. If speakers are not named, do not invent names;
say "the speaker" or "the group". If something is unclear, leave it out.

Return ONLY valid JSON with these keys:
"title": short neutral title in sentence case, using only words/topics from the excerpt.
"summary": a DETAILED factual summary of 2-4 paragraphs separated by blank lines (about 6-14 sentences, fewer if the excerpt is short).
  Cover every topic in the order discussed, including the specifics that were actually stated (figures, names, reasons, problems, options).
  Every sentence must be directly supported by the excerpt. Begin sentences with neutral openers such as "The speaker", "The group",
  "It was mentioned that", "They", "The discussion" - never with a person's name.
"key_points": [{"point": str, "quote": str}]
"decisions": [{"decision": str, "quote": str}] - ONLY something explicitly agreed, decided or approved ("we decided", "let's go with", "agreed").
  Ideas, suggestions, options, opinions, or "we should/might" are NOT decisions.
"action_items": [{"task": str, "owner": str, "due": str, "quote": str}] - ONLY when someone is clearly assigned a task or commits to it
  ("I will...", "X will...", "Can you...?" followed by agreement). owner = a name only if stated in the excerpt, else "Unassigned".
  due = the deadline exactly as spoken, else "N/A". Wishes, ideas, problems or general discussion are NOT action items.
"open_questions": [{"question": str, "quote": str}] - questions explicitly raised and left unresolved.
Every "quote" must be copied VERBATIM (max 25 words) from a single transcript line and must start with that line's [HH:MM:SS] timestamp.
If you cannot give a verbatim quote, omit the item. Empty lists are correct and expected when nothing qualifies. Never pad."""

FINAL_SYS = """You are given factual notes from consecutive parts of ONE meeting. Merge them into one detailed, coherent summary of 3-6 paragraphs
separated by blank lines. Use ONLY facts present in the notes: add nothing, infer nothing, keep names and numbers exactly as written.
Begin sentences with neutral openers ("The speaker", "The group", "It was mentioned that", "They") - never with a person's name.
Return ONLY JSON: {"title": short sentence-case title using only words from the notes, "summary": str}."""

def _email(r):
    if not r["summary"] and not (r["decisions"] or r["action_items"]): return ""
    L = ["Hi all,", "", "Thank you for joining. Here is a recap of what was discussed.", "", r["summary"]]
    if r["decisions"]: L += ["", "Decisions:"] + [f"- {d['text']} [{d['ts']}]" for d in r["decisions"]]
    if r["action_items"]: L += ["", "Action items:"] + [f"- {a['text']} (owner: {a['owner']}, due: {a['due']}) [{a['ts']}]" for a in r["action_items"]]
    if r["open_questions"]: L += ["", "Open questions:"] + [f"- {q['text']} [{q['ts']}]" for q in r["open_questions"]]
    return "\n".join(L + ["", "Best regards"])

def analyze_text(transcript, provider, key, model):
    tr = Transcript(transcript)
    if not tr.lines: raise ApiError("Transcript is empty.")
    parts = [parse_json(chat(provider, key, model, CHUNK_SYS, "TRANSCRIPT EXCERPT:\n" + ch, json_mode=True)) for ch in _chunks(tr)]
    sums = [as_text(p.get("summary")) for p in parts]
    summary, title = "\n\n".join(s for s in sums if s), as_text(parts[0].get("title"))
    joined = "\n\n".join(f"Part {i + 1}:\n{s}" for i, s in enumerate(sums) if s)
    if len(parts) > 1 and 0 < len(joined) <= 14000:
        try:
            m = parse_json(chat(provider, key, model, FINAL_SYS, joined, json_mode=True))
            if as_text(m.get("summary")): summary, title = as_text(m["summary"]), as_text(m.get("title")) or title
        except Exception:
            pass                                       # fall back to the concatenated (already grounded) part summaries
    summary, removed = tr.filter_text(summary)
    if not title or tr.ungrounded(title): title = "Meeting notes"
    pts, b1 = _take(tr, parts, "key_points", "point")
    dec, b2 = _take(tr, parts, "decisions", "decision", firm=True)
    act, b3 = _take(tr, parts, "action_items", "task", _action_extra, firm=True)
    qs, b4 = _take(tr, parts, "open_questions", "question")
    r = {"title": title, "summary": summary or "No verifiable summary could be produced from this transcript.",
         "key_points": pts, "decisions": dec, "action_items": act, "open_questions": qs, "removed": removed + b1 + b2 + b3 + b4}
    r["followup_email"] = _email(r)
    return r

# ---------- Rendering (same look, now with evidence) ----------
def _ev(x): return f'<div style="opacity:.65;font-size:12px;margin-top:2px">⏱ {esc(x["ts"])} · “{esc(x["quote"])}”</div>'
def _items(xs, empty="Nothing explicit in the transcript."):
    if not xs: return f'<div style="opacity:.6">{empty}</div>'
    return "".join(f'<div style="padding:7px 0;border-top:1px solid var(--border-color-primary)"><div>{esc(x["text"])}</div>{_ev(x)}</div>' for x in xs)

def render_cards(r):
    acts = "".join(
        '<div style="display:flex;gap:10px;align-items:flex-start;padding:8px 0;border-top:1px solid var(--border-color-primary)">'
        '<div style="width:16px;height:16px;border:2px solid #6366f1;border-radius:5px;margin-top:3px;flex:none"></div>'
        f'<div><div style="font-weight:500">{esc(a["text"])}</div><div style="margin-top:4px">'
        f'{chip("👤 " + a["owner"], "rgba(99,102,241,.18)")}{chip("📅 " + a["due"], "rgba(245,158,11,.22)")}</div>{_ev(a)}</div></div>'
        for a in r["action_items"]) or '<div style="opacity:.6">No clear assignment or commitment found in the transcript.</div>'
    paras = "".join(f'<p style="margin:0 0 10px;line-height:1.65">{esc(p)}</p>' for p in r["summary"].split("\n\n"))
    note = (f'🔒 Grounding check: {r["removed"]} unverifiable item(s)/sentence(s) were removed.' if r["removed"]
            else "🔒 Grounding check passed: every item is backed by a transcript quote.")
    return (f'<div><div style="font-size:22px;font-weight:700">{esc(r["title"])}</div>'
            f'<div style="opacity:.65;font-size:13px">{datetime.date.today():%d %b %Y} · {len(r["action_items"])} action items · {len(r["decisions"])} decisions</div></div>'
            + card("📝", "Summary", paras) + card("✅", "Action items", acts)
            + card("🏁", "Decisions", _items(r["decisions"])) + card("💡", "Key points", _items(r["key_points"]))
            + card("❓", "Open questions", _items(r["open_questions"], "None raised."))
            + f'<details style="{CARD}"><summary style="cursor:pointer;font-weight:600">✉️ Follow-up email</summary>'
              f'<pre style="white-space:pre-wrap;font-family:inherit;margin:10px 0 0">{esc(r["followup_email"])}</pre></details>'
            + f'<div style="opacity:.6;font-size:12px;margin-top:8px">{note}</div>')

def to_markdown(r):
    dec = "\n".join(f"- {d['text']} [{d['ts']}] — “{d['quote']}”" for d in r["decisions"]) or "- None stated"
    act = "\n".join(f"- [ ] {a['text']} - **{a['owner']}** (due: {a['due']}) [{a['ts']}] — “{a['quote']}”" for a in r["action_items"]) or "- None stated"
    kp = "\n".join(f"- {k['text']} [{k['ts']}]" for k in r["key_points"]) or "- None"
    oq = "\n".join(f"- {q['text']} [{q['ts']}]" for q in r["open_questions"]) or "- None"
    return (f"# {r['title']}\n_{datetime.date.today():%d %b %Y}_\n\n## Summary\n{r['summary']}\n\n## Action Items\n{act}\n\n"
            f"## Decisions\n{dec}\n\n## Key Points\n{kp}\n\n## Open Questions\n{oq}\n\n## Follow-up Email\n{r['followup_email']}\n")

print("Grounded pipeline patch applied.")

In [ ]:
import inspect
import gradio as gr

def pick(ui, name): return (ui or "").strip() or KEYS.get(name, "")
def groq_key(gk): return pick(gk, "GROQ_KEY")
def notes_key(provider, key, gk):
    if (key or "").strip(): return key.strip()
    if provider == "Groq": return groq_key(gk) or KEYS.get("LLM_KEY", "")   # reuse the transcription key
    return KEYS.get("LLM_KEY", "")

def status(kind, msg):
    c = {"ok": "rgba(34,197,94,.15)", "err": "rgba(239,68,68,.15)", "info": "rgba(99,102,241,.14)"}[kind]
    return f'<div style="background:{c};border-radius:10px;padding:10px 14px;margin:6px 0">{esc(msg)}</div>'

def make_notes(text, title, provider, key, gk, model):
    k = notes_key(provider, key, gk)
    if not k: raise ApiError("No notes key. Leave the key empty to reuse your Groq key, or add a key for the chosen provider.")
    r = analyze_text(text, provider, k, (model or "").strip() or PROVIDERS[provider]["model"])
    r["title"] = (title or "").strip() or r["title"]
    md = to_markdown(r)
    f = os.path.join(tempfile.gettempdir(), (re.sub(r"\W+", "_", r["title"]).strip("_") or "meeting") + ".md")
    with open(f, "w", encoding="utf-8") as fh: fh.write(md)
    return render_cards(r), md, f, r["title"]

def run_all(mic, file, gk, wm, lang, title, provider, key, model, progress=gr.Progress()):
    text = ""
    def out(kind, msg, cards=EMPTY, md="", f=None, t=title): return text, cards, md, f, t, status(kind, msg)
    path = getattr(file, "name", file) or mic
    if not path:
        yield out("err", "Record or upload a meeting first."); return
    try:
        yield out("info", "⏳ Transcribing audio...")
        text = transcribe_groq(path, groq_key(gk), wm, lang, lambda frac, m: progress(frac, desc=m))
        if not text.strip():
            yield out("err", "No speech detected (silent or unreadable audio)."); return
        yield out("info", "⏳ Writing notes...")
        progress(0.8, desc="Writing notes...")
        cards, md, f, t = make_notes(text, title, provider, key, gk, model)
        yield out("ok", "✅ Done - open the Notes tab, or ask questions in the Ask tab.", cards, md, f, t)
    except Exception as e:
        yield out("err", "❌ " + friendly(e))

def regen_fn(text, title, provider, key, model, gk, progress=gr.Progress()):
    if not (text or "").strip():
        yield EMPTY, "", None, title, status("err", "Transcript is empty."); return
    try:
        yield EMPTY, "", None, title, status("info", "⏳ Writing notes...")
        progress(0.5, desc="Writing notes...")
        cards, md, f, t = make_notes(text, title, provider, key, gk, model)
        yield cards, md, f, t, status("ok", "✅ Notes updated - see the Notes tab.")
    except Exception as e:
        yield EMPTY, "", None, title, status("err", "❌ " + friendly(e))

def ask_fn(q, transcript, provider, key, model, gk):
    if not (q or "").strip(): return "Type a question first."
    if not (transcript or "").strip(): return "No transcript yet - process a meeting in the Capture tab first."
    try:
        k = notes_key(provider, key, gk)
        if not k: raise ApiError("No notes key. Add one in Settings (or reuse your Groq key).")
        return chat(provider, k, (model or "").strip() or PROVIDERS[provider]["model"],
                    "Answer ONLY from the meeting transcript. Cite timestamps like [00:12:03]. If the answer is not there, say so.",
                    f"TRANSCRIPT:\n{pick_context(transcript, q)}\n\nQUESTION: {q}")
    except Exception as e:
        return "❌ " + friendly(e)

def share_fn(md, title, c_n, c_s, c_e, nt, nid, nk, sl, su, sp, rc):
    if not md: return "❌ Generate notes first."
    t = (title or "").strip() or "Meeting notes"
    jobs = []
    if c_n: jobs.append(("Notion", lambda: push_notion(pick(nt, "NOTION_TOKEN"), pick(nid, "NOTION_ID"), nk, t, md)))
    if c_s: jobs.append(("Slack", lambda: push_slack(pick(sl, "SLACK_WEBHOOK"), md)))
    if c_e: jobs.append(("Email", lambda: send_email(pick(su, "SMTP_USER"), pick(sp, "SMTP_PASS"), (rc or "").strip(), t, md)))
    if not jobs: return "Select at least one destination."
    log = []
    for name, fn in jobs:
        try: log.append(f"✅ **{name}**: {fn()}")
        except Exception as e: log.append(f"❌ **{name}**: {friendly(e)}")
    return "\n\n".join(log)

def lock(): return tuple(gr.Button(interactive=False) for _ in range(4))
def unlock(): return tuple(gr.Button(interactive=True) for _ in range(4))

# ---------- Gradio 5 / 6 compatible theme + css placement ----------
THEME = gr.themes.Soft(primary_hue="indigo", radius_size="lg")
CSS = ".gradio-container{max-width:1080px!important;margin:auto!important} footer{display:none!important}"
_new = {"theme", "css"} <= set(inspect.signature(gr.Blocks.launch).parameters)   # True on Gradio 6
blocks_kw = {} if _new else {"theme": THEME, "css": CSS}
launch_kw = {"theme": THEME, "css": CSS} if _new else {}
if KEYS.get("APP_PASSWORD"): launch_kw["auth"] = ("admin", KEYS["APP_PASSWORD"])
ph = lambda n: "✓ loaded from Colab Secrets / session" if KEYS.get(n) else ""

with gr.Blocks(title="AI Meeting Assistant", **blocks_kw) as demo:
    gr.HTML('<div style="text-align:center;padding:8px 0"><div style="font-size:28px;font-weight:700">🎙️ AI Meeting Assistant</div>'
            '<div style="opacity:.7">Transcribe → summarize → action items → share</div></div>')
    banner = gr.HTML(status("info", "Ready - record or upload a meeting."))
    md_state = gr.State("")

    with gr.Accordion("⚙️ Settings", open=False):
        with gr.Tabs():
            with gr.Tab("🔑 AI"):
                with gr.Row():
                    gkey = gr.Textbox(type="password", label="Groq API key (transcription)", placeholder=ph("GROQ_KEY"))
                    wmodel = gr.Dropdown(["whisper-large-v3-turbo", "whisper-large-v3"], value="whisper-large-v3-turbo", label="Whisper model")
                    lang = gr.Dropdown(["auto", "en", "hi", "te", "ta", "bn", "es", "fr", "de", "pt", "ar", "ja", "zh"], value="auto", label="Language")
                with gr.Row():
                    provider = gr.Dropdown(list(PROVIDERS), value="Groq", label="Notes LLM")
                    key = gr.Textbox(type="password", label="Notes API key (empty = reuse Groq key)", placeholder=ph("LLM_KEY"))
                    model = gr.Textbox(value=PROVIDERS["Groq"]["model"], label="Notes model")
                provider.change(lambda p: PROVIDERS[p]["model"], provider, model)
            with gr.Tab("🔗 Notion"):
                with gr.Row():
                    ntoken = gr.Textbox(type="password", label="Integration token", placeholder=ph("NOTION_TOKEN"))
                    nid = gr.Textbox(label="Page/database ID or URL", placeholder=ph("NOTION_ID"))
                    nkind = gr.Radio(["Page", "Database"], value="Page", label="Target type")
            with gr.Tab("💬 Slack"):
                slack = gr.Textbox(type="password", label="Incoming webhook URL", placeholder=ph("SLACK_WEBHOOK"))
            with gr.Tab("✉️ Email"):
                with gr.Row():
                    suser = gr.Textbox(label="Gmail address", placeholder=ph("SMTP_USER"))
                    spw = gr.Textbox(type="password", label="Gmail app password", placeholder=ph("SMTP_PASS"))
                    rcpt = gr.Textbox(label="Send notes to")

    with gr.Tabs():
        with gr.Tab("1 · Capture"):
            with gr.Row():
                mic = gr.Audio(sources=["microphone"], type="filepath", label="🎤 Record")
                file = gr.File(file_types=["audio", "video"], type="filepath", label="📁 Upload audio/video (mp3, wav, m4a, mp4...)")
            title = gr.Textbox(label="Meeting title (optional)")
            run = gr.Button("✨ Transcribe & Generate Notes", variant="primary", size="lg")
            transcript = gr.Textbox(label="Transcript (editable, or paste your own)", lines=12, max_lines=25)
            regen = gr.Button("🔄 Regenerate notes from transcript")
        with gr.Tab("2 · Notes"):
            cards = gr.HTML(EMPTY)
            dl = gr.File(label="⬇️ Download Markdown")
        with gr.Tab("3 · Ask"):
            q = gr.Textbox(label="Ask about the meeting", placeholder="What did we decide about the launch date?")
            ask_btn = gr.Button("Ask", variant="primary")
            ans = gr.Markdown()
        with gr.Tab("4 · Share"):
            with gr.Row():
                c1 = gr.Checkbox(label="Notion", value=True); c2 = gr.Checkbox(label="Slack"); c3 = gr.Checkbox(label="Email")
            send_btn = gr.Button("🚀 Send", variant="primary")
            share_out = gr.Markdown()

    BTNS = [run, regen, ask_btn, send_btn]
    def wire(btn, fn, ins, outs):
        btn.click(lock, None, BTNS, queue=False, show_progress="hidden") \
           .then(fn, ins, outs) \
           .then(unlock, None, BTNS, queue=False, show_progress="hidden")

    wire(run, run_all, [mic, file, gkey, wmodel, lang, title, provider, key, model], [transcript, cards, md_state, dl, title, banner])
    wire(regen, regen_fn, [transcript, title, provider, key, model, gkey], [cards, md_state, dl, title, banner])
    wire(ask_btn, ask_fn, [q, transcript, provider, key, model, gkey], ans)
    wire(send_btn, share_fn, [md_state, title, c1, c2, c3, ntoken, nid, nkind, slack, suser, spw, rcpt], share_out)

demo.queue().launch(share=True, debug=True, **launch_kw)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://abdb73ad265b750f8f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [12]:
!pip -q install -U gradio requests
import sys, gradio
print("Python", sys.version.split()[0], "| Gradio", gradio.__version__)

Python 3.13.15 | Gradio 6.29.0


In [13]:
import os, re, json, glob, time, shutil, smtplib, tempfile, subprocess, datetime, html, requests
from email.message import EmailMessage

esc = html.escape
KEYS = {}  # filled in Cell 3. Server-side only, never sent to the browser.

def sec(name):
    try:
        from google.colab import userdata
        return userdata.get(name) or ""
    except Exception:
        return os.environ.get(name, "")

class ApiError(Exception):
    pass

# ================= HTTP helper: retries + readable errors =================
def _msg(r):
    try:
        j = r.json()
        if isinstance(j, list) and j: j = j[0]
        e = j.get("error", j) if isinstance(j, dict) else j
        return (e.get("message") if isinstance(e, dict) else str(e)) or r.text[:300]
    except Exception:
        return r.text[:300]

HINTS = {401: "Invalid or missing API key.", 403: "Access denied for this key/model.",
         404: "Model or endpoint not found - check the model name.",
         413: "Input too large for this model/plan.", 429: "Rate limit reached."}

def _retry_after(r, default):
    try: return float(r.headers.get("retry-after", default))
    except Exception: return default

def post(url, label, retries=3, **kw):
    for n in range(retries + 1):
        try:
            r = requests.post(url, timeout=300, **kw)
        except requests.exceptions.RequestException as e:
            if n < retries:
                time.sleep(3 * (n + 1)); continue
            raise ApiError(f"{label}: network problem ({type(e).__name__}). Check your connection and retry.")
        if r.status_code in (429, 500, 502, 503) and n < retries:
            wait = _retry_after(r, 10 * (n + 1))
            if r.status_code == 429 and wait > 90:
                raise ApiError(f"{label}: rate limit reached - try again in ~{int(wait // 60) + 1} min. {_msg(r)}")
            time.sleep(min(wait, 90)); continue
        if not r.ok:
            raise ApiError(f"{label} error {r.status_code}: {HINTS.get(r.status_code, '')} {_msg(r)}".strip())
        return r

def friendly(e):
    if isinstance(e, ApiError): return str(e)
    if isinstance(e, requests.exceptions.RequestException):
        return f"Network problem ({type(e).__name__}). Check your connection and retry."
    if isinstance(e, smtplib.SMTPAuthenticationError):
        return "Gmail rejected the login. Use a 16-character App Password (needs 2-step verification), not your normal password."
    if isinstance(e, smtplib.SMTPException): return f"Email error: {e}"
    return f"Unexpected error: {type(e).__name__}: {e}"

def _model_problem(e):
    s = str(e).lower()
    return bool(re.search(r" (400|403|404)", s)) and any(w in s for w in ("model", "not found", "decommission", "access", "permission"))

# ================= Transcription (Groq Whisper API) + hallucination filtering =================
GROQ_STT = "https://api.groq.com/openai/v1/audio/transcriptions"
CHUNK_SEC = 600
HALLU = {"thank you", "thanks", "thank you very much", "thanks for watching", "thank you for watching",
         "please subscribe", "bye", "goodbye", "bye bye", "you", "okay", "ok"}

def norm(t): return re.sub(r"[^\w\s]", "", t.lower()).strip()
def is_hallu(n):
    return n in HALLU or n.startswith(("thanks for watching", "thank you for watching")) or "amaraorg" in n

def silences(path, db=-45, dur=1.0):
    """Silent spans (start, end) in seconds. Lower db (e.g. -50) = less aggressive."""
    p = subprocess.run(["ffmpeg", "-hide_banner", "-nostats", "-i", path, "-vn", "-af",
                        f"silencedetect=noise={db}dB:d={dur}", "-f", "null", "-"], capture_output=True, text=True)
    out, start = [], None
    for line in p.stderr.splitlines():
        m = re.search(r"silence_start:\s*(-?[\d.]+)", line)
        if m: start = max(0.0, float(m.group(1))); continue
        m = re.search(r"silence_end:\s*([\d.]+)", line)
        if m and start is not None: out.append((start, float(m.group(1)))); start = None
    if start is not None: out.append((start, 1e9))
    return out

def silent_fraction(a, b, spans):
    d = max(b - a, 1e-3)
    return min(1.0, sum(max(0, min(b, y) - max(a, x)) for x, y in spans) / d)

def clean_segments(segs, spans):
    kept = []
    for s in segs:
        text = s["text"].strip()
        if not text: continue
        if s["nsp"] > 0.6 and s["lp"] < -1.0: continue   # Whisper's own "no speech" rule
        if s["cr"] > 2.4: continue                       # repetitive gibberish
        s["n"] = norm(text); s["sil"] = silent_fraction(s["start"], s["end"], spans)
        if s["sil"] >= 0.9: continue
        if s["sil"] >= 0.5 and (is_hallu(s["n"]) or s["nsp"] > 0.3): continue
        if is_hallu(s["n"]) and s["nsp"] > 0.5: continue
        kept.append(s)
    out, run = [], 0                                     # repeat filter: max 2 identical lines in a row
    for s in kept:
        run = run + 1 if out and s["n"] == out[-1]["n"] else 1
        if run <= 2: out.append(s)
    j = len(out)                                         # trailing "thank you" style hallucinations
    while j > 0 and is_hallu(out[j - 1]["n"]): j -= 1
    tail = out[j:]
    if len(tail) >= 2 or (len(tail) == 1 and (tail[0]["sil"] > 0.3 or tail[0]["nsp"] > 0.3)):
        out = out[:j]
    return out

def format_segments(segs):
    return "\n".join(f"[{int(s['start'])//3600:02d}:{int(s['start'])%3600//60:02d}:{int(s['start'])%60:02d}] {s['text'].strip()}"
                     for s in segs)

def transcribe_groq(path, gkey, model, language, cb=lambda f, m: None):
    if not gkey: raise ApiError("No Groq API key. Add it in Settings (free: console.groq.com/keys).")
    if not shutil.which("ffmpeg"): raise ApiError("ffmpeg missing. Run  !apt-get -qq install ffmpeg  and retry.")
    tmp = tempfile.mkdtemp()
    try:
        p = subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", path, "-vn", "-ac", "1", "-ar", "16000",
                            "-b:a", "32k", "-f", "segment", "-segment_time", str(CHUNK_SEC),
                            "-reset_timestamps", "1", f"{tmp}/part_%03d.mp3"], capture_output=True, text=True)
        parts = sorted(glob.glob(f"{tmp}/part_*.mp3"))
        if p.returncode != 0 or not parts:
            raise ApiError("Could not read audio from this file (no audio track or unsupported format). " + p.stderr[-200:])
        cb(0.05, "Detecting silence...")
        spans = silences(path)
        models = [model] + [m for m in ("whisper-large-v3-turbo", "whisper-large-v3") if m != model]
        segs = []
        for i, part in enumerate(parts):
            cb(0.1 + 0.6 * i / len(parts), f"Transcribing part {i + 1}/{len(parts)}...")
            with open(part, "rb") as fh: blob = fh.read()
            r = None
            for m in list(models):
                data = {"model": m, "response_format": "verbose_json", "temperature": 0}
                if language != "auto": data["language"] = language
                try:
                    r = post(GROQ_STT, "Groq transcription", headers={"Authorization": f"Bearer {gkey}"},
                             files={"file": (os.path.basename(part), blob, "audio/mpeg")}, data=data)
                    models = [m] + [x for x in models if x != m]   # keep the working model first
                    break
                except ApiError as e:
                    if _model_problem(e): continue
                    raise
            if r is None: raise ApiError("No Whisper model is available for this key.")
            for s in r.json().get("segments", []):
                segs.append({"start": i * CHUNK_SEC + s.get("start", 0), "end": i * CHUNK_SEC + s.get("end", 0),
                             "text": s.get("text", ""), "nsp": s.get("no_speech_prob") or 0,
                             "lp": s.get("avg_logprob") or 0, "cr": s.get("compression_ratio") or 0})
        cb(0.72, "Cleaning transcript...")
        return format_segments(clean_segments(segs, spans))
    finally:
        shutil.rmtree(tmp, ignore_errors=True)

# ================= LLM (direct HTTP, no openai package) + self-healing model choice =================
PROVIDERS = {
    "Groq":   {"url": "https://api.groq.com/openai/v1", "model": "openai/gpt-oss-120b"},
    "OpenAI": {"url": "https://api.openai.com/v1", "model": "gpt-4o-mini"},
    "Gemini": {"url": "https://generativelanguage.googleapis.com/v1beta/openai", "model": "gemini-2.5-flash"},
}
PREFS = {
    "Groq":   ["openai/gpt-oss-120b", "openai/gpt-oss-20b", "qwen/qwen3.8-27b", "qwen/qwen3.6-27b", "llama-3.3-70b-versatile"],
    "OpenAI": ["gpt-4o-mini", "gpt-4.1-mini", "gpt-4o"],
    "Gemini": ["gemini-2.5-flash", "gemini-2.0-flash", "gemini-flash-latest"],
}
_SKIP = ("whisper", "guard", "tts", "embed", "orpheus", "image", "moderation", "transcribe", "audio", "safeguard", "dall", "realtime", "compound")
_MODELS, _WORKING = {}, {}

def list_models(provider, key, refresh=False):
    ck = (provider, key[-8:])
    if refresh or not _MODELS.get(ck):
        ids = []
        try:
            r = requests.get(PROVIDERS[provider]["url"].rstrip("/") + "/models",
                             headers={"Authorization": f"Bearer {key}"}, timeout=30)
            if r.ok:
                ids = [m["id"].replace("models/", "") for m in r.json().get("data", [])]
        except Exception:
            pass
        _MODELS[ck] = ids
    return _MODELS[ck]

def candidates(provider, key, requested):
    ids = list_models(provider, key)
    chat_ids = [i for i in ids if not any(s in i.lower() for s in _SKIP)]
    order = [_WORKING.get(provider), requested] + PREFS.get(provider, []) + chat_ids
    out = []
    for m in order:
        if m and m not in out and (not ids or m in ids):
            out.append(m)
    return (out or [requested])[:6]

def chat(provider, key, model, system, user, json_mode=False):
    if not key: raise ApiError(f"No API key for {provider}.")
    url = PROVIDERS[provider]["url"].rstrip("/") + "/chat/completions"
    hdr = {"Authorization": f"Bearer {key}", "Content-Type": "application/json"}
    last = None
    for m in candidates(provider, key, model):
        body = {"model": m, "temperature": 0,
                "messages": [{"role": "system", "content": system}, {"role": "user", "content": user}]}
        if json_mode: body["response_format"] = {"type": "json_object"}
        try:
            try:
                r = post(url, provider, headers=hdr, json=body)
            except ApiError as e:
                if json_mode and " 400" in str(e) and any(w in str(e).lower() for w in ("json", "response_format")):
                    body.pop("response_format"); r = post(url, provider, headers=hdr, json=body)
                else:
                    raise
        except ApiError as e:
            last = e
            if _model_problem(e):
                print(f"[info] {provider} model '{m}' unavailable -> trying next")
                continue
            raise
        if _WORKING.get(provider) != m:
            _WORKING[provider] = m; print(f"[info] using {provider} model: {m}")
        try:
            text = r.json()["choices"][0]["message"]["content"] or ""
        except Exception:
            raise ApiError(f"{provider}: unexpected response format.")
        return re.sub(r"<think>.*?</think>", "", text, flags=re.S).strip()
    ids = list_models(provider, key)
    raise ApiError(f"{provider}: none of the models worked ({last}). Models your key can see: "
                   f"{', '.join(ids[:15]) or 'none - the key may be invalid or the endpoint unreachable'}")

def parse_json(raw):
    raw = re.sub(r"^```(?:json)?|```$", "", raw.strip(), flags=re.M).strip()
    try: return json.loads(raw)
    except Exception:
        m = re.search(r"\{.*\}", raw, re.S)
        if m:
            try: return json.loads(m.group(0))
            except Exception: pass
    raise ApiError("The model did not return valid JSON. Click Regenerate or pick another model.")

def as_list(x): return [] if x is None else list(x) if isinstance(x, (list, tuple)) else [x]
def as_text(x): return " - ".join(str(v) for v in x.values() if v) if isinstance(x, dict) else str(x or "").strip()

# ================= Strictly transcript-grounded notes =================
TS_RE = re.compile(r"^\[(\d{1,2}:\d{2}:\d{2})\]\s*(.*)$")
WORD_RE = re.compile(r"[A-Za-z0-9][\w+#./'’-]*")
_UNITS = {w: i for i, w in enumerate("zero one two three four five six seven eight nine ten eleven twelve thirteen fourteen fifteen sixteen seventeen eighteen nineteen".split())}
_TENS = {"twenty": 20, "thirty": 30, "forty": 40, "fifty": 50, "sixty": 60, "seventy": 70, "eighty": 80, "ninety": 90}
OPENERS = set("""the a an it its they their he she we our this that these those there then also next first second third finally lastly
overall however additionally later earlier initially meanwhile afterward subsequently in on at after before during as when while according
several some one two three many most all both each speaker speakers discussion topics participants attendees group team meeting conversation
another other no not here if because since although though for from with without by to of about regarding concerning throughout among between
instead otherwise still yet so but and or now today i ok okay""".split())
STOP = set("that this with from have will would should could about there their were been they them then than what when which while also into over such just very more some".split())
_NONE = {"", "unassigned", "n/a", "na", "none", "unknown", "not specified", "not stated", "tbd"}
HEDGE = re.compile(r"\b(maybe|perhaps|might|possibly|someday|should we|could we|what if|if we|wish|hopefully)\b", re.I)

def _n(t): return " ".join(re.findall(r"[a-z0-9']+", t.lower().replace("’", "'")))
def _tok(t):
    t = t.lower().replace("’", "'").strip(".,;:!?)('\"")
    return t[:-2] if t.endswith("'s") else t

class Transcript:
    def __init__(self, text):
        self.lines = []
        for raw in text.splitlines():
            s = raw.strip()
            if not s: continue
            m = TS_RE.match(s)
            ts, t = (m.group(1), m.group(2)) if m else (None, s)
            self.lines.append((ts, t, _n(t)))
        self.has_ts = any(l[0] for l in self.lines)
        self.vocab = set()
        for _, t, _x in self.lines:
            self.vocab.update(_tok(w) for w in WORD_RE.findall(t))
        words = " ".join(l[2] for l in self.lines).split()
        for i, w in enumerate(words):                  # spoken numbers -> digits ("twenty five" -> 25)
            if w in _UNITS: self.vocab.add(str(_UNITS[w]))
            if w in _TENS:
                self.vocab.add(str(_TENS[w]))
                if i + 1 < len(words) and 0 < _UNITS.get(words[i + 1], 0) < 10:
                    self.vocab.add(str(_TENS[w] + _UNITS[words[i + 1]]))

    def locate(self, quote, hint=None):
        """Index of the transcript line containing the quote (verbatim, or >=85% word overlap), else None."""
        q = _n(re.sub(r"\[\d{1,2}:\d{2}:\d{2}\]", " ", quote or ""))
        qt = q.split()
        if len(qt) < 3: return None
        L = self.lines
        hits = [i for i, l in enumerate(L) if q in l[2]]
        if not hits:
            hits = [i for i in range(len(L) - 1) if q in L[i][2] + " " + L[i + 1][2]]
        if not hits and len(set(qt)) >= 4:
            qs = set(qt)
            for i in range(len(L)):
                win = set((L[i][2] + " " + (L[i + 1][2] if i + 1 < len(L) else "")).split())
                if len(qs & win) / len(qs) >= 0.85: hits.append(i)
        if not hits: return None
        m = re.search(r"\d{1,2}:\d{2}:\d{2}", hint or "")
        if m:
            for i in hits:
                if (L[i][0] or "").zfill(8) == m.group(0).zfill(8): return i
        return hits[0]

    def ctx(self, i, n=2):
        return " ".join(l[1] for l in self.lines[max(0, i - n): i + n + 2])

    def ungrounded(self, s, strict_first=False):
        """True if the text has a name / acronym / number that never appears in the transcript."""
        for k, t in enumerate(WORD_RE.findall(s)):
            n = _tok(t)
            if not n or n in OPENERS: continue
            dig = any(c.isdigit() for c in n)
            chk = dig or (t[0].isupper() if k else (strict_first and t[0].isupper()))
            if chk and n not in self.vocab:
                if "-" in n and all(p in self.vocab for p in n.split("-") if p): continue
                return True
        return False

    def filter_text(self, text):
        out, removed = [], 0
        for para in re.split(r"\n\s*\n", text or ""):
            keep = []
            for s in re.split(r"(?<=[.!?])\s+", para.strip()):
                if not s.strip(): continue
                if self.ungrounded(s, True): removed += 1
                else: keep.append(s.strip())
            if keep: out.append(" ".join(keep))
        return "\n\n".join(out), removed

def _supported(text, ctx_text):
    ws = [w for w in re.findall(r"[a-z0-9']{4,}", text.lower()) if w not in STOP]
    if not ws: return True
    cv = set(_n(ctx_text).split())
    hit = sum(1 for w in ws if w in cv or any(len(c) >= 4 and (c.startswith(w[:5]) or w.startswith(c[:5])) for c in cv))
    return hit / len(ws) >= 0.3

def _action_extra(it, tr, i):
    cs = set(_n(tr.ctx(i, 3)).split())
    owner, due = as_text(it.get("owner")), as_text(it.get("due"))
    ow, dw = _n(owner).split(), _n(due).split()
    if owner.lower() in _NONE or not ow or not all(w in cs for w in ow): owner = "Unassigned"
    if due.lower() in _NONE or not dw or not all(w in cs for w in dw): due = "N/A"
    return {"owner": owner, "due": due}

def _take(tr, parts, key, field, extra=None, firm=False):
    out, seen, bad = [], set(), 0
    for p in parts:
        for it in as_list(p.get(key)):
            if not isinstance(it, dict): bad += 1; continue
            text, quote = as_text(it.get(field)), as_text(it.get("quote"))
            i = tr.locate(quote, quote) if text and quote else None
            if i is None or tr.ungrounded(text) or not _supported(text, tr.ctx(i, 2)):
                bad += 1; continue
            if firm and HEDGE.search(tr.lines[i][1]): bad += 1; continue   # hedged talk is not a decision/commitment
            k = i if key == "action_items" else _n(text)[:90]               # one action item per transcript line
            if k in seen: continue
            seen.add(k)
            d = {"text": text, "ts": (tr.lines[i][0] if tr.has_ts else None) or "no timestamp", "quote": tr.lines[i][1][:220]}
            if extra: d.update(extra(it, tr, i))
            out.append(d)
    return out, bad

def _chunks(tr, limit=10000):
    out, cur, n = [], [], 0
    for ts, t, _x in tr.lines:
        line = f"[{ts}] {t}" if ts else t
        if cur and n + len(line) > limit: out.append("\n".join(cur)); cur, n = [], 0
        cur.append(line); n += len(line) + 1
    if cur: out.append("\n".join(cur))
    return out

CHUNK_SYS = """You are a meticulous, literal meeting note-taker. Use ONLY information explicitly stated in the transcript excerpt.
NEVER infer, guess, assume, generalize, or use outside knowledge. Do not add any person names, roles, job titles, companies, products,
tools, technologies, numbers, dates or deadlines that are not written in the excerpt. If speakers are not named, do not invent names;
say "the speaker" or "the group". If something is unclear, leave it out.

Return ONLY valid JSON with these keys:
"title": short neutral title in sentence case, using only words/topics from the excerpt.
"summary": a DETAILED factual summary of 2-4 paragraphs separated by blank lines (about 6-14 sentences, fewer if the excerpt is short).
  Cover every topic in the order discussed, including the specifics that were actually stated (figures, names, reasons, problems, options).
  Every sentence must be directly supported by the excerpt. Begin sentences with neutral openers such as "The speaker", "The group",
  "It was mentioned that", "They", "The discussion" - never with a person's name.
"key_points": [{"point": str, "quote": str}]
"decisions": [{"decision": str, "quote": str}] - ONLY something explicitly agreed, decided or approved ("we decided", "let's go with", "agreed").
  Ideas, suggestions, options, opinions, or "we should/might" are NOT decisions.
"action_items": [{"task": str, "owner": str, "due": str, "quote": str}] - ONLY when someone is clearly assigned a task or commits to it
  ("I will...", "X will...", "Can you...?" followed by agreement). owner = a name only if stated in the excerpt, else "Unassigned".
  due = the deadline exactly as spoken, else "N/A". Wishes, ideas, problems or general discussion are NOT action items.
"open_questions": [{"question": str, "quote": str}] - questions explicitly raised and left unresolved.
Every "quote" must be copied VERBATIM (max 25 words) from a single transcript line and must start with that line's [HH:MM:SS] timestamp.
If you cannot give a verbatim quote, omit the item. Empty lists are correct and expected when nothing qualifies. Never pad."""

FINAL_SYS = """You are given factual notes from consecutive parts of ONE meeting. Merge them into one detailed, coherent summary of 3-6 paragraphs
separated by blank lines. Use ONLY facts present in the notes: add nothing, infer nothing, keep names and numbers exactly as written.
Begin sentences with neutral openers ("The speaker", "The group", "It was mentioned that", "They") - never with a person's name.
Return ONLY JSON: {"title": short sentence-case title using only words from the notes, "summary": str}."""

def _email(r):
    if not r["summary"] and not (r["decisions"] or r["action_items"]): return ""
    L = ["Hi all,", "", "Thank you for joining. Here is a recap of what was discussed.", "", r["summary"]]
    if r["decisions"]: L += ["", "Decisions:"] + [f"- {d['text']} [{d['ts']}]" for d in r["decisions"]]
    if r["action_items"]: L += ["", "Action items:"] + [f"- {a['text']} (owner: {a['owner']}, due: {a['due']}) [{a['ts']}]" for a in r["action_items"]]
    if r["open_questions"]: L += ["", "Open questions:"] + [f"- {q['text']} [{q['ts']}]" for q in r["open_questions"]]
    return "\n".join(L + ["", "Best regards"])

def analyze_text(transcript, provider, key, model):
    tr = Transcript(transcript)
    if not tr.lines: raise ApiError("Transcript is empty.")
    parts = [parse_json(chat(provider, key, model, CHUNK_SYS, "TRANSCRIPT EXCERPT:\n" + ch, json_mode=True)) for ch in _chunks(tr)]
    sums = [as_text(p.get("summary")) for p in parts]
    summary, title = "\n\n".join(s for s in sums if s), as_text(parts[0].get("title"))
    joined = "\n\n".join(f"Part {i + 1}:\n{s}" for i, s in enumerate(sums) if s)
    if len(parts) > 1 and 0 < len(joined) <= 14000:
        try:
            m = parse_json(chat(provider, key, model, FINAL_SYS, joined, json_mode=True))
            if as_text(m.get("summary")): summary, title = as_text(m["summary"]), as_text(m.get("title")) or title
        except Exception:
            pass                                       # fall back to the concatenated (already grounded) part summaries
    summary, removed = tr.filter_text(summary)
    if not title or tr.ungrounded(title): title = "Meeting notes"
    pts, b1 = _take(tr, parts, "key_points", "point")
    dec, b2 = _take(tr, parts, "decisions", "decision", firm=True)
    act, b3 = _take(tr, parts, "action_items", "task", _action_extra, firm=True)
    qs, b4 = _take(tr, parts, "open_questions", "question")
    r = {"title": title, "summary": summary or "No verifiable summary could be produced from this transcript.",
         "key_points": pts, "decisions": dec, "action_items": act, "open_questions": qs, "removed": removed + b1 + b2 + b3 + b4}
    r["followup_email"] = _email(r)
    return r

def pick_context(transcript, question, limit=12000):
    if len(transcript) <= limit: return transcript
    lines = transcript.splitlines()
    blocks = ["\n".join(lines[i:i + 12]) for i in range(0, len(lines), 12)]
    words = set(re.findall(r"\w{3,}", question.lower()))
    order = sorted(range(len(blocks)), key=lambda i: -len(words & set(re.findall(r"\w{3,}", blocks[i].lower()))))
    chosen, total = [], 0
    for i in order:
        if total + len(blocks[i]) <= limit: chosen.append(i); total += len(blocks[i])
    return "\n...\n".join(blocks[i] for i in sorted(chosen)) or blocks[0][:limit]

# ================= Rendering: cards + markdown =================
CARD = ("background:var(--background-fill-secondary);border:1px solid var(--border-color-primary);"
        "border-radius:14px;padding:16px 18px;margin:10px 0;color:var(--body-text-color)")
EMPTY = '<div style="opacity:.6;padding:12px">Notes will appear here.</div>'

def chip(t, bg): return f'<span style="background:{bg};border-radius:999px;padding:2px 10px;font-size:12px;margin-right:6px;white-space:nowrap">{esc(t)}</span>'
def card(icon, head, body): return f'<div style="{CARD}"><div style="font-weight:600;font-size:15px;margin-bottom:8px">{icon} {esc(head)}</div>{body}</div>'
def _ev(x): return f'<div style="opacity:.65;font-size:12px;margin-top:2px">⏱ {esc(x["ts"])} · “{esc(x["quote"])}”</div>'
def _items(xs, empty="Nothing explicit in the transcript."):
    if not xs: return f'<div style="opacity:.6">{empty}</div>'
    return "".join(f'<div style="padding:7px 0;border-top:1px solid var(--border-color-primary)"><div>{esc(x["text"])}</div>{_ev(x)}</div>' for x in xs)

def render_cards(r):
    acts = "".join(
        '<div style="display:flex;gap:10px;align-items:flex-start;padding:8px 0;border-top:1px solid var(--border-color-primary)">'
        '<div style="width:16px;height:16px;border:2px solid #6366f1;border-radius:5px;margin-top:3px;flex:none"></div>'
        f'<div><div style="font-weight:500">{esc(a["text"])}</div><div style="margin-top:4px">'
        f'{chip("👤 " + a["owner"], "rgba(99,102,241,.18)")}{chip("📅 " + a["due"], "rgba(245,158,11,.22)")}</div>{_ev(a)}</div></div>'
        for a in r["action_items"]) or '<div style="opacity:.6">No clear assignment or commitment found in the transcript.</div>'
    paras = "".join(f'<p style="margin:0 0 10px;line-height:1.65">{esc(p)}</p>' for p in r["summary"].split("\n\n"))
    note = (f'🔒 Grounding check: {r["removed"]} unverifiable item(s)/sentence(s) were removed.' if r["removed"]
            else "🔒 Grounding check passed: every item is backed by a transcript quote.")
    return (f'<div><div style="font-size:22px;font-weight:700">{esc(r["title"])}</div>'
            f'<div style="opacity:.65;font-size:13px">{datetime.date.today():%d %b %Y} · {len(r["action_items"])} action items · {len(r["decisions"])} decisions</div></div>'
            + card("📝", "Summary", paras) + card("✅", "Action items", acts)
            + card("🏁", "Decisions", _items(r["decisions"])) + card("💡", "Key points", _items(r["key_points"]))
            + card("❓", "Open questions", _items(r["open_questions"], "None raised."))
            + f'<details style="{CARD}"><summary style="cursor:pointer;font-weight:600">✉️ Follow-up email</summary>'
              f'<pre style="white-space:pre-wrap;font-family:inherit;margin:10px 0 0">{esc(r["followup_email"])}</pre></details>'
            + f'<div style="opacity:.6;font-size:12px;margin-top:8px">{note}</div>')

def to_markdown(r):
    dec = "\n".join(f"- {d['text']} [{d['ts']}] — “{d['quote']}”" for d in r["decisions"]) or "- None stated"
    act = "\n".join(f"- [ ] {a['text']} - **{a['owner']}** (due: {a['due']}) [{a['ts']}] — “{a['quote']}”" for a in r["action_items"]) or "- None stated"
    kp = "\n".join(f"- {k['text']} [{k['ts']}]" for k in r["key_points"]) or "- None"
    oq = "\n".join(f"- {q['text']} [{q['ts']}]" for q in r["open_questions"]) or "- None"
    return (f"# {r['title']}\n_{datetime.date.today():%d %b %Y}_\n\n## Summary\n{r['summary']}\n\n## Action Items\n{act}\n\n"
            f"## Decisions\n{dec}\n\n## Key Points\n{kp}\n\n## Open Questions\n{oq}\n\n## Follow-up Email\n{r['followup_email']}\n")

# ================= Integrations =================
def notion_call(method, url, token, **kw):
    r = requests.request(method, url, headers={"Authorization": f"Bearer {token}", "Notion-Version": "2022-06-28"}, timeout=60, **kw)
    if not r.ok:
        hint = {401: "Invalid Notion token.", 404: "Not found - share the page/database with your integration (⋯ → Connections)."}.get(r.status_code, "")
        raise ApiError(f"Notion error {r.status_code}: {hint} {_msg(r)}".strip())
    return r.json()

def push_notion(token, target, kind, title, md):
    if not token or not target: raise ApiError("Notion: add the integration token and page/database ID in Settings.")
    ids = re.findall(r"[0-9a-fA-F]{32}", (target.split("?")[0]).replace("-", ""))
    if not ids: raise ApiError("Notion: could not find a 32-character ID in what you pasted.")
    tid = ids[-1]
    rt = lambda t: [{"type": "text", "text": {"content": t[:1900]}}]
    blocks = []
    for line in md.splitlines():
        s = line.strip().replace("**", "")
        if len(s) > 2 and s[0] == "_" and s[-1] == "_": s = s[1:-1]
        if not s or s.startswith("# "): continue
        if s.startswith("## "): blocks.append({"type": "heading_2", "heading_2": {"rich_text": rt(s[3:])}})
        elif s.startswith("- [ ] "): blocks.append({"type": "to_do", "to_do": {"rich_text": rt(s[6:]), "checked": False}})
        elif s.startswith("- "): blocks.append({"type": "bulleted_list_item", "bulleted_list_item": {"rich_text": rt(s[2:])}})
        else:
            for i in range(0, len(s), 1900):
                blocks.append({"type": "paragraph", "paragraph": {"rich_text": rt(s[i:i + 1900])}})
    if kind == "Database":
        db = notion_call("GET", f"https://api.notion.com/v1/databases/{tid}", token)
        tp = next((k for k, v in db.get("properties", {}).items() if v.get("type") == "title"), None)
        if not tp: raise ApiError("Notion: that ID is not a database. Choose 'Page' as target type.")
        parent, props = {"database_id": tid}, {tp: {"title": rt(title)}}
    else:
        parent, props = {"page_id": tid}, {"title": {"title": rt(title)}}
    page = notion_call("POST", "https://api.notion.com/v1/pages", token,
                       json={"parent": parent, "properties": props, "children": blocks[:100]})
    for i in range(100, len(blocks), 100):
        notion_call("PATCH", f"https://api.notion.com/v1/blocks/{page['id']}/children", token, json={"children": blocks[i:i + 100]})
    return page.get("url", "created")

def push_slack(webhook, md):
    if not webhook: raise ApiError("Slack: add the webhook URL in Settings.")
    text = re.sub(r"^#+\s*(.*)$", r"*\1*", md, flags=re.M).replace("**", "*")[:35000]
    r = requests.post(webhook, json={"text": text}, timeout=30)
    if r.text.strip() != "ok": raise ApiError(f"Slack error {r.status_code}: {r.text[:200]}")
    return "sent"

def send_email(user, pw, to, subject, body):
    if not (user and pw and to): raise ApiError("Email: fill Gmail address, app password and recipient.")
    m = EmailMessage(); m["From"], m["To"], m["Subject"] = user, to, subject; m.set_content(body)
    with smtplib.SMTP_SSL("smtp.gmail.com", 465, timeout=30) as s:
        s.login(user, pw.replace(" ", "")); s.send_message(m)
    return f"sent to {to}"

print("Core loaded. Default notes model:", PROVIDERS["Groq"]["model"])

Core loaded. Default notes model: openai/gpt-oss-120b


In [14]:
from getpass import getpass

for k in ["GROQ_KEY", "LLM_KEY", "NOTION_TOKEN", "NOTION_ID", "SLACK_WEBHOOK", "SMTP_USER", "SMTP_PASS", "APP_PASSWORD"]:
    KEYS[k] = sec(k)
if not KEYS["GROQ_KEY"]:
    KEYS["GROQ_KEY"] = getpass("Groq API key (hidden, free at console.groq.com/keys) - Enter to skip: ").strip()

print("ffmpeg :", "OK" if shutil.which("ffmpeg") else "MISSING -> run: !apt-get -qq install ffmpeg")

if KEYS["GROQ_KEY"]:
    gk = KEYS["GROQ_KEY"]
    ids = list_models("Groq", gk, refresh=True)
    if ids:
        print("Models your key can see:", ", ".join(ids))
        print("Whisper turbo available:", "whisper-large-v3-turbo" in ids)
    else:
        try:
            r = requests.get(PROVIDERS["Groq"]["url"] + "/models", headers={"Authorization": f"Bearer {gk}"}, timeout=30)
            print(f"Could not list models -> HTTP {r.status_code}: {r.text[:300]}")
        except Exception as e:
            print("Could not reach Groq:", type(e).__name__, e)
    try:
        out = chat("Groq", gk, PROVIDERS["Groq"]["model"], "Reply with the single word OK.", "ping")
        print("Groq chat completions : OK ->", out.strip()[:20], "| model in use:", _WORKING.get("Groq"))
    except Exception as e:
        print("Groq chat completions : FAILED ->", friendly(e))
else:
    print("No Groq key entered - you can paste it inside the UI instead.")

Groq API key (hidden, free at console.groq.com/keys) - Enter to skip: ··········
ffmpeg : OK
Models your key can see: openai/gpt-oss-safeguard-20b, canopylabs/orpheus-v1-english, whisper-large-v3-turbo, qwen/qwen3.8-27b, whisper-large-v3, meta-llama/llama-prompt-guard-2-86m, openai/gpt-oss-120b, openai/gpt-oss-20b, allam-2-7b, canopylabs/orpheus-arabic-saudi, meta-llama/llama-prompt-guard-2-22m
Whisper turbo available: True
[info] using Groq model: openai/gpt-oss-120b
Groq chat completions : OK -> OK | model in use: openai/gpt-oss-120b


In [ ]:
import inspect
import gradio as gr

def pick(ui, name): return (ui or "").strip() or KEYS.get(name, "")
def groq_key(gk): return pick(gk, "GROQ_KEY")
def notes_key(provider, key, gk):
    if (key or "").strip(): return key.strip()
    if provider == "Groq": return groq_key(gk) or KEYS.get("LLM_KEY", "")   # reuse the transcription key
    return KEYS.get("LLM_KEY", "")

def status(kind, msg):
    c = {"ok": "rgba(34,197,94,.15)", "err": "rgba(239,68,68,.15)", "info": "rgba(99,102,241,.14)"}[kind]
    return f'<div style="background:{c};border-radius:10px;padding:10px 14px;margin:6px 0">{esc(msg)}</div>'

def make_notes(text, title, provider, key, gk, model):
    k = notes_key(provider, key, gk)
    if not k: raise ApiError("No notes key. Leave the key empty to reuse your Groq key, or add a key for the chosen provider.")
    r = analyze_text(text, provider, k, (model or "").strip() or PROVIDERS[provider]["model"])
    r["title"] = (title or "").strip() or r["title"]
    md = to_markdown(r)
    f = os.path.join(tempfile.gettempdir(), (re.sub(r"\W+", "_", r["title"]).strip("_") or "meeting") + ".md")
    with open(f, "w", encoding="utf-8") as fh: fh.write(md)
    return render_cards(r), md, f, r["title"]

def run_all(mic, file, gk, wm, lang, title, provider, key, model, progress=gr.Progress()):
    text = ""
    def out(kind, msg, cards=EMPTY, md="", f=None, t=title): return text, cards, md, f, t, status(kind, msg)
    path = getattr(file, "name", file) or mic
    if not path:
        yield out("err", "Record or upload a meeting first."); return
    try:
        yield out("info", "⏳ Transcribing audio...")
        text = transcribe_groq(path, groq_key(gk), wm, lang, lambda frac, m: progress(frac, desc=m))
        if not text.strip():
            yield out("err", "No speech detected (silent or unreadable audio)."); return
        yield out("info", "⏳ Writing notes...")
        progress(0.8, desc="Writing notes...")
        cards, md, f, t = make_notes(text, title, provider, key, gk, model)
        yield out("ok", "✅ Done - open the Notes tab, or ask questions in the Ask tab.", cards, md, f, t)
    except Exception as e:
        yield out("err", "❌ " + friendly(e))

def regen_fn(text, title, provider, key, model, gk, progress=gr.Progress()):
    if not (text or "").strip():
        yield EMPTY, "", None, title, status("err", "Transcript is empty."); return
    try:
        yield EMPTY, "", None, title, status("info", "⏳ Writing notes...")
        progress(0.5, desc="Writing notes...")
        cards, md, f, t = make_notes(text, title, provider, key, gk, model)
        yield cards, md, f, t, status("ok", "✅ Notes updated - see the Notes tab.")
    except Exception as e:
        yield EMPTY, "", None, title, status("err", "❌ " + friendly(e))

def ask_fn(q, transcript, provider, key, model, gk):
    if not (q or "").strip(): return "Type a question first."
    if not (transcript or "").strip(): return "No transcript yet - process a meeting in the Capture tab first."
    try:
        k = notes_key(provider, key, gk)
        if not k: raise ApiError("No notes key. Add one in Settings (or reuse your Groq key).")
        return chat(provider, k, (model or "").strip() or PROVIDERS[provider]["model"],
                    "Answer ONLY from the meeting transcript. Cite timestamps like [00:12:03]. If the answer is not there, say so. Never add outside knowledge.",
                    f"TRANSCRIPT:\n{pick_context(transcript, q)}\n\nQUESTION: {q}")
    except Exception as e:
        return "❌ " + friendly(e)

def share_fn(md, title, c_n, c_s, c_e, nt, nid, nk, sl, su, sp, rc):
    if not md: return "❌ Generate notes first."
    t = (title or "").strip() or "Meeting notes"
    jobs = []
    if c_n: jobs.append(("Notion", lambda: push_notion(pick(nt, "NOTION_TOKEN"), pick(nid, "NOTION_ID"), nk, t, md)))
    if c_s: jobs.append(("Slack", lambda: push_slack(pick(sl, "SLACK_WEBHOOK"), md)))
    if c_e: jobs.append(("Email", lambda: send_email(pick(su, "SMTP_USER"), pick(sp, "SMTP_PASS"), (rc or "").strip(), t, md)))
    if not jobs: return "Select at least one destination."
    log = []
    for name, fn in jobs:
        try: log.append(f"✅ **{name}**: {fn()}")
        except Exception as e: log.append(f"❌ **{name}**: {friendly(e)}")
    return "\n\n".join(log)

def lock(): return tuple(gr.Button(interactive=False) for _ in range(4))
def unlock(): return tuple(gr.Button(interactive=True) for _ in range(4))

# ---------- Gradio 5 / 6 compatible theme + css placement ----------
THEME = gr.themes.Soft(primary_hue="indigo", radius_size="lg")
CSS = ".gradio-container{max-width:1080px!important;margin:auto!important} footer{display:none!important}"
_new = {"theme", "css"} <= set(inspect.signature(gr.Blocks.launch).parameters)   # True on Gradio 6
blocks_kw = {} if _new else {"theme": THEME, "css": CSS}
launch_kw = {"theme": THEME, "css": CSS} if _new else {}
if KEYS.get("APP_PASSWORD"): launch_kw["auth"] = ("admin", KEYS["APP_PASSWORD"])
ph = lambda n: "✓ loaded from Colab Secrets / session" if KEYS.get(n) else ""

with gr.Blocks(title="AI Meeting Assistant", **blocks_kw) as demo:
    gr.HTML('<div style="text-align:center;padding:8px 0"><div style="font-size:28px;font-weight:700">🎙️ AI Meeting Assistant</div>'
            '<div style="opacity:.7">Transcribe → summarize → action items → share</div></div>')
    banner = gr.HTML(status("info", "Ready - record or upload a meeting."))
    md_state = gr.State("")

    with gr.Accordion("⚙️ Settings", open=False):
        with gr.Tabs():
            with gr.Tab("🔑 AI"):
                with gr.Row():
                    gkey = gr.Textbox(type="password", label="Groq API key (transcription)", placeholder=ph("GROQ_KEY"))
                    wmodel = gr.Dropdown(["whisper-large-v3-turbo", "whisper-large-v3"], value="whisper-large-v3-turbo", label="Whisper model")
                    lang = gr.Dropdown(["auto", "en", "hi", "te", "ta", "bn", "es", "fr", "de", "pt", "ar", "ja", "zh"], value="auto", label="Language")
                with gr.Row():
                    provider = gr.Dropdown(list(PROVIDERS), value="Groq", label="Notes LLM")
                    key = gr.Textbox(type="password", label="Notes API key (empty = reuse Groq key)", placeholder=ph("LLM_KEY"))
                    model = gr.Textbox(value=PROVIDERS["Groq"]["model"], label="Notes model (auto-falls back if unavailable)")
                provider.change(lambda p: PROVIDERS[p]["model"], provider, model)
            with gr.Tab("🔗 Notion"):
                with gr.Row():
                    ntoken = gr.Textbox(type="password", label="Integration token", placeholder=ph("NOTION_TOKEN"))
                    nid = gr.Textbox(label="Page/database ID or URL", placeholder=ph("NOTION_ID"))
                    nkind = gr.Radio(["Page", "Database"], value="Page", label="Target type")
            with gr.Tab("💬 Slack"):
                slack = gr.Textbox(type="password", label="Incoming webhook URL", placeholder=ph("SLACK_WEBHOOK"))
            with gr.Tab("✉️ Email"):
                with gr.Row():
                    suser = gr.Textbox(label="Gmail address", placeholder=ph("SMTP_USER"))
                    spw = gr.Textbox(type="password", label="Gmail app password", placeholder=ph("SMTP_PASS"))
                    rcpt = gr.Textbox(label="Send notes to")

    with gr.Tabs():
        with gr.Tab("1 · Capture"):
            with gr.Row():
                mic = gr.Audio(sources=["microphone"], type="filepath", label="🎤 Record")
                file = gr.File(file_types=["audio", "video"], type="filepath", label="📁 Upload audio/video (mp3, wav, m4a, mp4...)")
            title = gr.Textbox(label="Meeting title (optional)")
            run = gr.Button("✨ Transcribe & Generate Notes", variant="primary", size="lg")
            transcript = gr.Textbox(label="Transcript (editable, or paste your own)", lines=12, max_lines=25)
            regen = gr.Button("🔄 Regenerate notes from transcript")
        with gr.Tab("2 · Notes"):
            cards = gr.HTML(EMPTY)
            dl = gr.File(label="⬇️ Download Markdown")
        with gr.Tab("3 · Ask"):
            q = gr.Textbox(label="Ask about the meeting", placeholder="What did we decide about the launch date?")
            ask_btn = gr.Button("Ask", variant="primary")
            ans = gr.Markdown()
        with gr.Tab("4 · Share"):
            with gr.Row():
                c1 = gr.Checkbox(label="Notion", value=True); c2 = gr.Checkbox(label="Slack"); c3 = gr.Checkbox(label="Email")
            send_btn = gr.Button("🚀 Send", variant="primary")
            share_out = gr.Markdown()

    BTNS = [run, regen, ask_btn, send_btn]
    def wire(btn, fn, ins, outs):
        btn.click(lock, None, BTNS, queue=False, show_progress="hidden") \
           .then(fn, ins, outs) \
           .then(unlock, None, BTNS, queue=False, show_progress="hidden")

    wire(run, run_all, [mic, file, gkey, wmodel, lang, title, provider, key, model], [transcript, cards, md_state, dl, title, banner])
    wire(regen, regen_fn, [transcript, title, provider, key, model, gkey], [cards, md_state, dl, title, banner])
    wire(ask_btn, ask_fn, [q, transcript, provider, key, model, gkey], ans)
    wire(send_btn, share_fn, [md_state, title, c1, c2, c3, ntoken, nid, nkind, slack, suser, spw, rcpt], share_out)

demo.queue().launch(share=True, debug=True, **launch_kw)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://df4ffb03f591b29563.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
